# COMEX Precious Metals EFP Beta Analysis — Gold & Silver

> **Quantifying residual spot delta when long the EFP using Bloomberg BQL on BQuant**
>
> **Objective:** Quantify the delta (beta to spot) implicitly carried when
> long the COMEX Gold or Silver EFP (Exchange for Physical).
>
> **EFP = COMEX Futures Price - OTC Forward Price (to FND)**
>
> This notebook uses **generic futures tickers** (GC1-GC4, SI1-SI4) so it
> auto-updates without maintaining a specific contract list.
> Coverage: front month through approximately 3-9 months forward.
>
> Self-contained BQuant notebook — executable top-to-bottom.

## Table of Contents

### Prompt A — Data Pipeline
- [Section 0 — Imports and Configuration](#Section-0-—-Imports-and-Configuration)
- [Section 1 — Define Generic Ticker Lists](#Section-1-—-Define-Generic-Ticker-Lists)
- [Section 2 — BQL Pull 1: Contract Metadata](#Section-2-—-BQL-Pull-1:-Contract-Metadata-from-Generic-Tickers)
- [Section 3 — BQL Pull 2: Historical Prices](#Section-3-—-BQL-Pull-2:-Historical-Prices-for-Generic-Tickers)
- [Section 4 — BQL Pull 3: Spot Prices](#Section-4-—-BQL-Pull-3:-Gold-and-Silver-Spot-Prices)
- [Section 5 — BQL Pull 4: Forward Rate Curves](#Section-5-—-BQL-Pull-4:-Bloomberg-Native-Forward-Rate-Curves)
- [Section 6 — Delta\_T Calculation](#Section-6-—-Delta_T-Calculation-for-Generic-Tickers)
- [Section 7 — Master DataFrame Assembly](#Section-7-—-Master-DataFrame-Assembly)

### Prompt B — EFP Construction
- [Section 8 — Forward Rate Interpolation](#Section-8-—-Forward-Rate-Interpolation-to-Delta_T)
- [Section 9 — OTC Forward Price](#Section-9-—-OTC-Forward-Price-Anchored-to-FND)
- [Section 10 — EFP Series Construction](#Section-10-—-EFP-Series-Construction)
- [Section 11 — Daily Differenced Series](#Section-11-—-Daily-Differenced-Series-for-Regression)
- [Section 12 — EFP Time Series Visualisations](#Section-12-—-EFP-Time-Series-Visualisations)

### Prompt C — Regression & Beta
- [Section 13 — Static OLS Regression](#Section-13-—-Static-OLS-Regression:-Empirical-Beta)
- [Section 14 — Rolling OLS: Time-Varying Beta](#Section-14-—-Rolling-OLS:-Time-Varying-Beta)
- [Section 15 — Theoretical Beta](#Section-15-—-Theoretical-Beta-from-Cost-of-Carry)
- [Section 16 — Regression Diagnostics](#Section-16-—-Regression-Diagnostics)
- [Section 17 — Practical Delta Exposure](#Section-17-—-Practical-Interpretation:-Delta-Exposure)

### Prompt D — Dashboard & Documentation
- [Section 18 — Core Visualisation Suite](#Section-18-—-Core-Visualisation-Suite)
- [Section 19 — Summary Dashboard](#Section-19-—-Summary-Dashboard)
- [Section 20 — Sensitivity Analysis](#Section-20-—-Sensitivity-Analysis)
- [Section 21 — Daily Workflow Documentation](#Section-21-—-HOW-TO-USE-THIS-NOTEBOOK:-Daily-Workflow)
- [Section 23 — End-to-End Validation](#Section-23-—-End-to-End-Validation:-Worked-Example)

## Section 0 — Imports and Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pandas.tseries.offsets import BDay
from datetime import date, datetime

# Bloomberg BQL
import bql
bq = bql.Service()

# ── Configuration ────────────────────────────────────────────────
config = {
    'metals':                ['gold', 'silver'],
    'start_date':            '2023-01-01',
    'end_date':              'today',
    'roll_days_before_fnd':  5,
    'regression_window_days': 60,
    'k_sigma':               3,
    'generic_depth':         4,    # GC1-GC4, SI1-SI4
}

# Set end_date dynamically
config['end_date'] = date.today().strftime('%Y-%m-%d')

print(f"Session started : {datetime.now():%Y-%m-%d %H:%M}")
print(f"BQL service     : {type(bq).__name__}")
print(f"NumPy {np.__version__}  |  pandas {pd.__version__}")
print(f"\nConfiguration:")
for k, v in config.items():
    print(f"  {k:30s}: {v}")

## Section 1 — Define Generic Ticker Lists

Generic tickers auto-roll: GC1 always points to the current front-month
Gold contract, GC2 to the next, etc. No manual contract maintenance needed.

In [ ]:
# ── Generic futures tickers ────────────────────────────────────────
gold_generics   = [f'GC{i} Comdty' for i in range(1, config['generic_depth'] + 1)]
silver_generics = [f'SI{i} Comdty' for i in range(1, config['generic_depth'] + 1)]
all_generics    = gold_generics + silver_generics

# ── Metadata map ─────────────────────────────────────────────────
generic_meta = {}
for i in range(1, config['generic_depth'] + 1):
    generic_meta[f'GC{i} Comdty'] = {'metal': 'gold',   'position': i}
    generic_meta[f'SI{i} Comdty'] = {'metal': 'silver', 'position': i}

# ── Spot tickers ─────────────────────────────────────────────────
spot_tickers = {
    'gold':   'XAU Curncy',
    'silver': 'XAG Curncy',
}

# ── Forward rate curve tickers ───────────────────────────────────
fwd_ticker_map = {
    'gold': {
        '1W':  ('XAUUSD1W BGN Curncy',  7/365),
        '1M':  ('XAUUSD1M BGN Curncy',  1/12),
        '2M':  ('XAUUSD2M BGN Curncy',  2/12),
        '3M':  ('XAUUSD3M BGN Curncy',  3/12),
        '6M':  ('XAUUSD6M BGN Curncy',  6/12),
        '12M': ('XAUUSD12M BGN Curncy', 1.0),
    },
    'silver': {
        '1W':  ('XAGUSD1W BGN Curncy',  7/365),
        '1M':  ('XAGUSD1M BGN Curncy',  1/12),
        '2M':  ('XAGUSD2M BGN Curncy',  2/12),
        '3M':  ('XAGUSD3M BGN Curncy',  3/12),
        '6M':  ('XAGUSD6M BGN Curncy',  6/12),
        '12M': ('XAGUSD12M BGN Curncy', 1.0),
    },
}

print(f"Gold generics:   {gold_generics}")
print(f"Silver generics: {silver_generics}")
print(f"Spot tickers:    {spot_tickers}")
print(f"Forward tenors:  {list(fwd_ticker_map['gold'].keys())}")
print(f"\nTotal tickers to fetch: {len(all_generics)} generics + "
      f"2 spot + 12 forwards = {len(all_generics) + 2 + 12}")

## Section 2 — BQL Pull 1: Contract Metadata from Generic Tickers

Pull **static** fields for all 8 generic tickers. These reflect the
**current** active contract mapped to each generic and auto-update
each time the notebook is run.

> `fut_cur_gen_ticker()` tells us which specific contract (e.g. GCM25)
> is currently the front month. This updates automatically when Bloomberg
> rolls the generic. `fut_notice_first()` returns that contract's FND.
> These are the live anchors for all delta\_T calculations in this notebook.

In [ ]:
# ── Pull static contract metadata for all generics ────────────────
meta_rows = []

for ticker in all_generics:
    info = generic_meta[ticker]
    try:
        req = bql.Request(
            ticker,
            {
                'fnd':           bq.data.fut_notice_first(),
                'fdd':           bq.data.fut_dlv_dt_first(),
                'ltd':           bq.data.last_tradeable_dt(),
                'specific':      bq.data.fut_cur_gen_ticker(),
                'contract_size': bq.data.fut_contract_size(),
            }
        )
        resp = bq.execute(req)
        df = resp[0].df()

        meta_rows.append({
            'generic_ticker':    ticker,
            'specific_contract': df['specific'].iloc[0] if 'specific' in df.columns else None,
            'metal':             info['metal'],
            'position':          info['position'],
            'fnd':               pd.to_datetime(df['fnd'].iloc[0]) if 'fnd' in df.columns else pd.NaT,
            'fdd':               pd.to_datetime(df['fdd'].iloc[0]) if 'fdd' in df.columns else pd.NaT,
            'ltd':               pd.to_datetime(df['ltd'].iloc[0]) if 'ltd' in df.columns else pd.NaT,
            'contract_size_oz':  df['contract_size'].iloc[0] if 'contract_size' in df.columns else np.nan,
        })
        print(f"  {ticker:15s}  OK  -> {meta_rows[-1]['specific_contract']}")

    except Exception as exc:
        print(f"  {ticker:15s}  FAIL: {exc}")
        meta_rows.append({
            'generic_ticker': ticker, 'specific_contract': None,
            'metal': info['metal'], 'position': info['position'],
            'fnd': pd.NaT, 'fdd': pd.NaT, 'ltd': pd.NaT,
            'contract_size_oz': np.nan,
        })

contracts_meta = pd.DataFrame(meta_rows)

# Days to FND
contracts_meta['days_to_fnd'] = (
    contracts_meta['fnd'] - pd.Timestamp.today()
).dt.days

print("\n" + "=" * 80)
print("  CONTRACT METADATA (live snapshot)")
print("=" * 80)
display(contracts_meta[[
    'generic_ticker', 'specific_contract', 'metal', 'position',
    'fnd', 'days_to_fnd', 'contract_size_oz'
]])

## Section 3 — BQL Pull 2: Historical Prices for Generic Tickers

Pull daily settlement prices for all 8 generic tickers over the full
date range. With generic tickers, Bloomberg automatically handles the
roll — the price series is continuous.

Roll dates are detected heuristically: days where the daily price
change exceeds 3× the 20-day rolling standard deviation.

In [ ]:
# ── Pull settlement prices for all generics ───────────────────────
price_frames = []

for ticker in all_generics:
    info = generic_meta[ticker]
    print(f"  Fetching {ticker:15s} ...", end=" ")

    try:
        # Primary: PX_SETTLE
        req_settle = bql.Request(
            ticker,
            {'settle': bq.data.px_settle(
                dates=bq.func.range(config['start_date'], config['end_date']),
                fill='prev'
            )}
        )
        resp_settle = bq.execute(req_settle)
        df_settle = resp_settle[0].df()

        if df_settle.empty:
            print("empty response")
            continue

        df_settle = df_settle.set_index('DATE')
        df_settle.index = pd.to_datetime(df_settle.index)
        df_settle = df_settle[~df_settle.index.duplicated(keep='last')]
        df_settle = df_settle.sort_index()

        # Fallback: PX_LAST where settle is NaN
        req_last = bql.Request(
            ticker,
            {'px_last': bq.data.px_last(
                dates=bq.func.range(config['start_date'], config['end_date']),
                fill='prev'
            )}
        )
        resp_last = bq.execute(req_last)
        df_last = resp_last[0].df()
        if not df_last.empty:
            df_last = df_last.set_index('DATE')
            df_last.index = pd.to_datetime(df_last.index)
            df_last = df_last[~df_last.index.duplicated(keep='last')]
            df_last = df_last.sort_index()

            # Fill NaN in settle with px_last
            settle_col = df_settle.columns[0] if len(df_settle.columns) > 0 else 'settle'
            last_col = df_last.columns[0] if len(df_last.columns) > 0 else 'px_last'

            combined = pd.DataFrame({
                'settle': df_settle.iloc[:, 0],
                'px_last': df_last.iloc[:, 0],
            })
            combined['price'] = combined['settle'].fillna(combined['px_last'])
        else:
            combined = pd.DataFrame({'price': df_settle.iloc[:, 0]})

        combined = combined[['price']].dropna()

        for dt, row in combined.iterrows():
            price_frames.append({
                'date':     dt,
                'ticker':   ticker,
                'metal':    info['metal'],
                'position': info['position'],
                'settle':   row['price'],
            })

        print(f"OK  {len(combined):>5d} obs")

    except Exception as exc:
        print(f"FAIL: {exc}")

futures_prices = pd.DataFrame(price_frames)
futures_prices['date'] = pd.to_datetime(futures_prices['date'])
futures_prices = futures_prices.sort_values(['metal', 'position', 'date']).reset_index(drop=True)

# ── Detect roll dates heuristically ──────────────────────────────
futures_prices['is_roll_date'] = False

for ticker in all_generics:
    mask = futures_prices['ticker'] == ticker
    sub = futures_prices.loc[mask, 'settle'].copy()
    if len(sub) < 25:
        continue
    daily_chg = sub.diff().abs()
    rolling_std = sub.diff().rolling(20, min_periods=10).std()
    # Flag where |change| > 3x rolling std
    roll_mask = daily_chg > (config['k_sigma'] * rolling_std)
    # Map back
    futures_prices.loc[mask, 'is_roll_date'] = roll_mask.values

n_rolls = futures_prices['is_roll_date'].sum()

print(f"\nFutures prices: {futures_prices.shape[0]:,} rows")
print(f"Date range: {futures_prices['date'].min():%Y-%m-%d} to "
      f"{futures_prices['date'].max():%Y-%m-%d}")
print(f"Roll dates detected: {n_rolls}")
print(f"\nSample (GC1):")
display(futures_prices[futures_prices['ticker'] == 'GC1 Comdty'].head(5))

## Section 4 — BQL Pull 3: Gold and Silver Spot Prices

Pull daily `PX_LAST` for XAU and XAG spot over the full date range.

In [ ]:
# ── Pull spot prices ──────────────────────────────────────────────
spot_frames = []

for metal, ticker in spot_tickers.items():
    print(f"  Fetching {ticker:15s} ({metal}) ...", end=" ")
    try:
        req = bql.Request(
            ticker,
            {'spot': bq.data.px_last(
                dates=bq.func.range(config['start_date'], config['end_date']),
                fill='prev'
            )}
        )
        resp = bq.execute(req)
        df = resp[0].df()

        if df.empty:
            print("empty response")
            continue

        df = df.set_index('DATE')
        df.index = pd.to_datetime(df.index)
        df = df[~df.index.duplicated(keep='last')]
        df = df.sort_index()

        for dt, row in df.iterrows():
            spot_frames.append({
                'date':  dt,
                'metal': metal,
                'spot':  row.iloc[0],
            })

        print(f"OK  {len(df):>5d} obs")

    except Exception as exc:
        print(f"FAIL: {exc}")

spot_prices = pd.DataFrame(spot_frames)
spot_prices['date'] = pd.to_datetime(spot_prices['date'])
spot_prices = spot_prices.dropna(subset=['spot'])
spot_prices = spot_prices.sort_values(['metal', 'date']).reset_index(drop=True)

print(f"\nSpot prices: {spot_prices.shape[0]:,} rows")
for metal in config['metals']:
    sub = spot_prices[spot_prices['metal'] == metal]
    if len(sub) > 0:
        print(f"  {metal:8s}: {len(sub):>5d} obs  "
              f"[{sub['date'].min():%Y-%m-%d} -> {sub['date'].max():%Y-%m-%d}]  "
              f"last=${sub['spot'].iloc[-1]:,.2f}")

## Section 5 — BQL Pull 4: Bloomberg Native Forward Rate Curves

These are all-in annualised % rates embedding USD rates, lease rates,
and storage. We use them directly — **do NOT reconstruct from SOFR**.

Tenors: 1W, 1M, 2M, 3M, 6M, 12M for both Gold and Silver.

In [ ]:
# ── Pull forward rate curves ──────────────────────────────────────
fwd_frames = []

for metal, tenors in fwd_ticker_map.items():
    for tenor_label, (ticker, tenor_years) in tenors.items():
        print(f"  {ticker:28s} ({metal} {tenor_label:>3s}) ...", end=" ")
        try:
            req = bql.Request(
                ticker,
                {'rate': bq.data.px_last(
                    dates=bq.func.range(config['start_date'], config['end_date']),
                    fill='prev'
                )}
            )
            resp = bq.execute(req)
            df = resp[0].df()

            if df.empty:
                print("empty")
                continue

            df = df.set_index('DATE')
            df.index = pd.to_datetime(df.index)
            df = df[~df.index.duplicated(keep='last')]
            df = df.sort_index()

            for dt, row in df.iterrows():
                fwd_frames.append({
                    'date':        dt,
                    'metal':       metal,
                    'tenor_label': tenor_label,
                    'tenor_years': tenor_years,
                    'rate_pct':    row.iloc[0],
                })

            print(f"OK  {len(df):>5d} obs")

        except Exception as exc:
            print(f"FAIL: {exc}")

fwd_rates = pd.DataFrame(fwd_frames)
fwd_rates['date'] = pd.to_datetime(fwd_rates['date'])
fwd_rates['rate_dec'] = fwd_rates['rate_pct'] / 100.0
fwd_rates = fwd_rates.dropna(subset=['rate_pct'])
fwd_rates = fwd_rates.sort_values(['metal', 'tenor_years', 'date']).reset_index(drop=True)

print(f"\nForward rates: {fwd_rates.shape[0]:,} rows")
for metal in config['metals']:
    sub = fwd_rates[fwd_rates['metal'] == metal]
    n_tenors = sub['tenor_label'].nunique()
    print(f"  {metal:8s}: {len(sub):>6d} obs across {n_tenors} tenors  "
          f"[{sub['date'].min():%Y-%m-%d} -> {sub['date'].max():%Y-%m-%d}]")

# Show latest snapshot
print("\nLatest forward rate snapshot:")
latest_date = fwd_rates['date'].max()
latest = fwd_rates[fwd_rates['date'] == latest_date]
for metal in config['metals']:
    sub = latest[latest['metal'] == metal].sort_values('tenor_years')
    print(f"\n  {metal.upper()} ({latest_date:%Y-%m-%d}):")
    for _, row in sub.iterrows():
        print(f"    {row['tenor_label']:>4s}  ({row['tenor_years']:.4f}y)  "
              f"rate = {row['rate_pct']:+.4f}%")

## Section 6 — Delta\_T Calculation for Generic Tickers

With generic tickers, delta\_T (time-to-FND in years) must be inferred
for each historical date.

**Delivery months:**
- Gold: Feb, Apr, Jun, Aug, Oct, Dec
- Silver: Mar, May, Jul, Sep, Dec

**FND convention:** Last business day of the month **prior** to the
delivery month.

Step 6a computes the live delta\_T from `contracts_meta`.
Step 6b applies the historical FND inference to all rows in `futures_prices`.

In [ ]:
# ── Delivery months ───────────────────────────────────────────────
DELIVERY_MONTHS = {
    'gold':   [2, 4, 6, 8, 10, 12],
    'silver': [3, 5, 7, 9, 12],
}


def get_fnd_for_date(dt, metal, position, roll_buffer_days=5):
    """Infer the FND for a given date, metal, and generic position.

    FND = last business day of the month PRIOR to the delivery month.
    Position 1 = front month, position 2 = next delivery month, etc.

    Parameters
    ----------
    dt : datetime-like
        The observation date.
    metal : str
        'gold' or 'silver'.
    position : int
        Generic position (1 = front, 2 = second, etc.).
    roll_buffer_days : int
        Skip contracts where days_to_fnd < this value.

    Returns
    -------
    pd.Timestamp
        The inferred FND for the contract at this position.
    """
    dt = pd.Timestamp(dt)
    delivery_months = DELIVERY_MONTHS[metal]

    # Build list of upcoming delivery months starting from dt
    candidates = []
    year = dt.year
    for y_offset in range(0, 3):  # look up to 3 years out
        for m in delivery_months:
            # FND = last bday of month prior to delivery month
            if m == 1:
                fnd_month = 12
                fnd_year = year + y_offset - 1
            else:
                fnd_month = m - 1
                fnd_year = year + y_offset

            # Last business day of fnd_month
            fnd_eom = pd.Timestamp(fnd_year, fnd_month, 1) + pd.offsets.MonthEnd(0)
            fnd = fnd_eom
            # Adjust to last business day
            while fnd.weekday() >= 5:  # Saturday=5, Sunday=6
                fnd -= pd.Timedelta(days=1)

            days_to = (fnd - dt).days
            if days_to >= roll_buffer_days:
                candidates.append(fnd)

    candidates.sort()

    if position <= len(candidates):
        return candidates[position - 1]
    elif len(candidates) > 0:
        return candidates[-1]
    else:
        return pd.NaT


# ── Step 6a: Live delta_T from contracts_meta ────────────────────
print("=" * 70)
print("  STEP 6a — LIVE DELTA_T (from contracts_meta)")
print("=" * 70)
today = pd.Timestamp.today().normalize()

for _, row in contracts_meta.iterrows():
    if pd.notna(row['fnd']):
        delta_T = (row['fnd'] - today).days / 365
        print(f"  {row['generic_ticker']:15s}  "
              f"contract={row['specific_contract']:10s}  "
              f"FND={row['fnd']:%Y-%m-%d}  "
              f"delta_T={delta_T:.4f}y  ({row['days_to_fnd']}d)")

# ── Step 6b: Historical delta_T for all rows ─────────────────────
print("\n" + "=" * 70)
print("  STEP 6b — HISTORICAL DELTA_T (inferred FND)")
print("=" * 70)

futures_prices['fnd'] = futures_prices.apply(
    lambda r: get_fnd_for_date(
        r['date'], r['metal'], r['position'],
        roll_buffer_days=config['roll_days_before_fnd']
    ),
    axis=1
)

futures_prices['delta_T'] = (
    futures_prices['fnd'] - futures_prices['date']
).dt.days / 365.0

futures_prices['near_expiry'] = futures_prices['delta_T'] < (5 / 365)

# Sanity check stats
for metal in config['metals']:
    for pos in range(1, config['generic_depth'] + 1):
        sub = futures_prices[
            (futures_prices['metal'] == metal) &
            (futures_prices['position'] == pos)
        ]
        if len(sub) > 0:
            print(f"  {metal:8s} pos={pos}  "
                  f"delta_T range=[{sub['delta_T'].min():.4f}, {sub['delta_T'].max():.4f}]  "
                  f"near_expiry={sub['near_expiry'].sum()} days")

print(f"\nTotal rows with delta_T: {futures_prices['delta_T'].notna().sum():,} "
      f"/ {len(futures_prices):,}")

### Delta\_T Sawtooth Check (GC1)

The front-month delta\_T should exhibit a sawtooth pattern: decreasing
within each contract (as FND approaches) and jumping up at each roll
(when the generic switches to the next delivery month).

In [ ]:
# ── Sawtooth plot for GC1 ─────────────────────────────────────────
gc1 = futures_prices[
    (futures_prices['ticker'] == 'GC1 Comdty') &
    (futures_prices['delta_T'].notna())
].copy()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(gc1['date'], gc1['delta_T'], linewidth=0.8, color='#2c3e50')
ax.axhline(y=0, color='red', linestyle='--', linewidth=0.5, alpha=0.5)

# Mark roll dates
rolls = gc1[gc1['is_roll_date']]
ax.scatter(rolls['date'], rolls['delta_T'], color='red', s=15, zorder=5,
           label=f'Roll dates ({len(rolls)})')

ax.set_title('GC1 Delta_T (years to FND) — Sawtooth Pattern', fontsize=12)
ax.set_ylabel('Delta_T (years)')
ax.set_xlabel('')
ax.legend(loc='upper right')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Same for SI1
si1 = futures_prices[
    (futures_prices['ticker'] == 'SI1 Comdty') &
    (futures_prices['delta_T'].notna())
].copy()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(si1['date'], si1['delta_T'], linewidth=0.8, color='#7f8c8d')
ax.axhline(y=0, color='red', linestyle='--', linewidth=0.5, alpha=0.5)

rolls_si = si1[si1['is_roll_date']]
ax.scatter(rolls_si['date'], rolls_si['delta_T'], color='red', s=15, zorder=5,
           label=f'Roll dates ({len(rolls_si)})')

ax.set_title('SI1 Delta_T (years to FND) — Sawtooth Pattern', fontsize=12)
ax.set_ylabel('Delta_T (years)')
ax.legend(loc='upper right')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"GC1: {len(gc1)} data points, {len(rolls)} roll dates")
print(f"SI1: {len(si1)} data points, {len(rolls_si)} roll dates")

## Section 7 — Master DataFrame Assembly

Join futures (front month only), spot, forward rates, and second-generic
prices into a single `master_df` for downstream analysis.

Columns: `date, metal, ticker, futures_price, futures_price_g2, spot,
fnd, delta_T, fwd_1W, fwd_1M, fwd_2M, fwd_3M, fwd_6M, fwd_12M,
is_roll_date, near_expiry`

In [ ]:
# ── Base: front-month (position=1) futures ────────────────────────
front = futures_prices[futures_prices['position'] == 1][[
    'date', 'metal', 'ticker', 'settle', 'fnd', 'delta_T',
    'is_roll_date', 'near_expiry'
]].rename(columns={'settle': 'futures_price'}).copy()

# ── Add second-generic (position=2) for calendar spread ──────────
g2 = futures_prices[futures_prices['position'] == 2][[
    'date', 'metal', 'settle'
]].rename(columns={'settle': 'futures_price_g2'}).copy()

master_df = front.merge(g2, on=['date', 'metal'], how='left')

# ── Left join spot prices ────────────────────────────────────────
master_df = master_df.merge(spot_prices, on=['date', 'metal'], how='left')

# ── Left join forward rates (pivot to wide) ──────────────────────
fwd_pivot = fwd_rates.pivot_table(
    index=['date', 'metal'],
    columns='tenor_label',
    values='rate_dec',
    aggfunc='last'
).reset_index()

# Rename tenor columns with fwd_ prefix
tenor_cols = [c for c in fwd_pivot.columns if c not in ('date', 'metal')]
fwd_rename = {t: f'fwd_{t}' for t in tenor_cols}
fwd_pivot = fwd_pivot.rename(columns=fwd_rename)

master_df = master_df.merge(fwd_pivot, on=['date', 'metal'], how='left')

# ── Drop rows with missing critical data ─────────────────────────
pre_drop = len(master_df)
master_df = master_df.dropna(subset=['futures_price', 'spot'])
post_drop = len(master_df)
print(f"Dropped {pre_drop - post_drop} rows with NaN futures_price or spot")

master_df = master_df.sort_values(['metal', 'date']).reset_index(drop=True)

# ── Data quality summary ─────────────────────────────────────────
print("\n" + "=" * 80)
print("  MASTER DATAFRAME — DATA QUALITY SUMMARY")
print("=" * 80)
print(f"  Shape: {master_df.shape[0]:,} rows x {master_df.shape[1]} columns")

for metal in config['metals']:
    sub = master_df[master_df['metal'] == metal]
    if len(sub) == 0:
        continue
    n_rolls = sub['is_roll_date'].sum()
    latest = sub.iloc[-1]
    # Look up specific contract from contracts_meta
    g1_meta = contracts_meta[
        (contracts_meta['metal'] == metal) & (contracts_meta['position'] == 1)
    ]
    specific = g1_meta['specific_contract'].iloc[0] if len(g1_meta) > 0 else '?'

    print(f"\n  {metal.upper()}:")
    print(f"    Date range       : {sub['date'].min():%Y-%m-%d} to {sub['date'].max():%Y-%m-%d}")
    print(f"    Observations     : {len(sub):,}")
    print(f"    Roll dates       : {n_rolls}")
    print(f"    Current contract : {specific}")
    print(f"    Current delta_T  : {latest['delta_T']:.4f} years ({latest['delta_T']*365:.0f} days)")
    print(f"    Current FND      : {latest['fnd']}")
    print(f"    Futures price    : ${latest['futures_price']:,.2f}")
    print(f"    Spot price       : ${latest['spot']:,.2f}")
    print(f"    Basis (F-S)      : ${latest['futures_price'] - latest['spot']:+,.2f}")

# Forward rate coverage
fwd_cols = [c for c in master_df.columns if c.startswith('fwd_')]
print(f"\n  Forward rate columns: {fwd_cols}")
for col in fwd_cols:
    pct_filled = master_df[col].notna().mean() * 100
    print(f"    {col:12s}: {pct_filled:.1f}% filled")

print(f"\n  Columns: {list(master_df.columns)}")
display(master_df.head(5))

### Live Status

In [ ]:
# ── Print live status ─────────────────────────────────────────────
print("=" * 80)
print(f"  LIVE STATUS — as of {date.today():%Y-%m-%d}")
print("=" * 80)

for metal in config['metals']:
    g1 = contracts_meta[
        (contracts_meta['metal'] == metal) & (contracts_meta['position'] == 1)
    ]
    if len(g1) == 0:
        continue
    row = g1.iloc[0]
    latest = master_df[master_df['metal'] == metal].iloc[-1]
    delta_T = latest['delta_T']

    print(f"\n  {metal.upper()} front month:")
    print(f"    Specific contract : {row['specific_contract']}")
    print(f"    FND               : {row['fnd']:%Y-%m-%d}")
    print(f"    Days to FND       : {row['days_to_fnd']}")
    print(f"    Delta_T           : {delta_T:.4f} years")
    print(f"    Futures price     : ${latest['futures_price']:,.2f}")
    print(f"    Spot price        : ${latest['spot']:,.2f}")
    print(f"    Basis (F-S)       : ${latest['futures_price'] - latest['spot']:+,.2f}")
    print(f"    Basis (%)         : {(latest['futures_price']/latest['spot'] - 1)*100:+.3f}%")

### Export Master DataFrame

In [ ]:
# ── Save master_df ────────────────────────────────────────────────
output_path = 'efp_master_data.csv'
master_df.to_csv(output_path, index=False)
print(f"Saved master_df to: {output_path}")
print(f"  {master_df.shape[0]:,} rows x {master_df.shape[1]} columns")

# Also save contracts metadata
meta_path = 'efp_contracts_meta.csv'
contracts_meta.to_csv(meta_path, index=False)
print(f"Saved contracts_meta to: {meta_path}")

print(f"\nPrompt A complete: {datetime.now():%Y-%m-%d %H:%M}")

---
# Prompt B — Forward Rate Interpolation, OTC Forward, EFP Construction

> Sections 8–12: Interpolate the forward rate curve to each contract's
> exact delta\_T, construct the OTC forward price anchored to FND,
> compute EFP series (raw and adjusted), and build differenced series
> for regression in Prompt C.

In [ ]:
# ── Reload guard: load from CSV if master_df not in memory ────────
try:
    _ = master_df.shape
    print(f"master_df already in memory: {master_df.shape}")
except NameError:
    import pandas as pd
    import numpy as np
    from scipy.interpolate import interp1d
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    import seaborn as sns
    from datetime import date, datetime
    import warnings
    warnings.filterwarnings("ignore")

    master_df = pd.read_csv('efp_master_data.csv', parse_dates=['date', 'fnd'])
    contracts_meta = pd.read_csv('efp_contracts_meta.csv',
                                  parse_dates=['fnd', 'fdd', 'ltd'])
    print(f"Loaded master_df from CSV: {master_df.shape}")
    print(f"Loaded contracts_meta from CSV: {contracts_meta.shape}")

    # Rebuild config if not present
    config = {
        'metals': ['gold', 'silver'],
        'start_date': '2023-01-01',
        'end_date': date.today().strftime('%Y-%m-%d'),
        'roll_days_before_fnd': 5,
        'regression_window_days': 60,
        'k_sigma': 3,
        'generic_depth': 4,
    }

## Section 8 — Forward Rate Interpolation to Delta\_T

The forward rate curve has fixed tenors (1W, 1M, 2M, 3M, 6M, 12M).
Each futures contract has its own delta\_T (time to FND in years) that
generally falls **between** these tenors.

We interpolate the curve to each row's exact delta\_T using `scipy.interp1d`
with flat extrapolation at the edges.

Edge cases handled:
- delta\_T <= 0 → rate = 0 (at or past FND)
- < 2 non-NaN rate points → NaN (insufficient data)
- delta\_T < 1W → use 1W rate directly (no extrapolation below shortest tenor)

In [ ]:
def interpolate_fwd_rate(row, method='linear'):
    """Interpolate the forward rate curve to this row's exact delta_T.

    Parameters
    ----------
    row : pd.Series
        A row from master_df with fwd_1W...fwd_12M and delta_T.
    method : str
        Interpolation kind passed to scipy.interpolate.interp1d.

    Returns
    -------
    float
        Interpolated forward rate as a decimal.
    """
    delta_T = row.get('delta_T', np.nan)

    # Edge case: at or past FND
    if pd.isna(delta_T) or delta_T <= 0:
        return 0.0

    # Build tenor/rate arrays from the row's forward columns
    tenors_all = np.array([7/365, 1/12, 2/12, 3/12, 6/12, 12/12])
    rates_all  = np.array([
        row.get('fwd_1W', np.nan),
        row.get('fwd_1M', np.nan),
        row.get('fwd_2M', np.nan),
        row.get('fwd_3M', np.nan),
        row.get('fwd_6M', np.nan),
        row.get('fwd_12M', np.nan),
    ])

    # Drop NaN pairs
    valid = ~np.isnan(rates_all)
    tenors = tenors_all[valid]
    rates  = rates_all[valid]

    if len(rates) < 2:
        return np.nan

    # Edge case: delta_T shorter than 1W — use shortest available rate
    if delta_T < tenors[0]:
        return float(rates[0])

    # Interpolate
    interp_fn = interp1d(
        tenors, rates,
        kind=method,
        bounds_error=False,
        fill_value=(rates[0], rates[-1]),  # flat extrapolation at edges
    )

    return float(interp_fn(delta_T))


# ── Apply to master_df ───────────────────────────────────────────
master_df['fwd_rate_interp'] = master_df.apply(interpolate_fwd_rate, axis=1)

n_nan = master_df['fwd_rate_interp'].isna().sum()
n_total = len(master_df)
print(f"Forward rate interpolation complete:")
print(f"  {n_total - n_nan:,} rows interpolated, {n_nan} NaN ({n_nan/n_total*100:.1f}%)")

# ── Per-metal summary ────────────────────────────────────────────
print("\n" + "=" * 70)
print("  INTERPOLATED FORWARD RATE SUMMARY")
print("=" * 70)
for metal in config['metals']:
    sub = master_df[master_df['metal'] == metal]['fwd_rate_interp'].dropna()
    if len(sub) == 0:
        continue
    print(f"\n  {metal.upper()}:")
    print(f"    Mean  : {sub.mean()*100:+.4f}%")
    print(f"    Median: {sub.median()*100:+.4f}%")
    print(f"    Std   : {sub.std()*100:.4f}%")
    print(f"    Range : [{sub.min()*100:+.4f}%, {sub.max()*100:+.4f}%]")
    print(f"    NaN   : {master_df[master_df['metal']==metal]['fwd_rate_interp'].isna().sum()}")

### Interpolation Sanity Check

Plot the interpolated rate overlaid with raw 1M and 3M rates.
The interpolated rate should track between them when delta\_T
is in the 1–3 month range.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for i, metal in enumerate(config['metals']):
    ax = axes[i]
    sub = master_df[master_df['metal'] == metal].copy()
    sub = sub.sort_values('date')

    ax.plot(sub['date'], sub['fwd_rate_interp'] * 100,
            linewidth=1.2, color='#2c3e50', label='Interpolated (at delta_T)')

    # Overlay raw tenors
    if 'fwd_1M' in sub.columns:
        ax.plot(sub['date'], sub['fwd_1M'] * 100,
                linewidth=0.8, color='#3498db', alpha=0.6, label='1M raw')
    if 'fwd_3M' in sub.columns:
        ax.plot(sub['date'], sub['fwd_3M'] * 100,
                linewidth=0.8, color='#e67e22', alpha=0.6, label='3M raw')

    ax.set_title(f'{metal.upper()} — Interpolated Forward Rate vs Raw Tenors',
                 fontsize=12)
    ax.set_ylabel('Rate (%)')
    ax.legend(loc='best', fontsize=9)
    ax.axhline(y=0, color='grey', linestyle='--', linewidth=0.5)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Flag NaN rows
nan_rows = master_df[master_df['fwd_rate_interp'].isna()]
if len(nan_rows) > 0:
    print(f"\nWARNING: {len(nan_rows)} rows with NaN interpolated rate:")
    print(nan_rows[['date', 'metal', 'delta_T', 'fwd_1M', 'fwd_3M']].head(10))
else:
    print("\nAll rows have valid interpolated forward rates.")

## Section 9 — OTC Forward Price Anchored to FND

The OTC forward is the theoretical price at which a dealer would sell
gold/silver for delivery on FND, given today's spot and the interpolated
forward rate:

$$F_{OTC} = S \times (1 + r_{fwd} \times \Delta T)$$

This uses **simple interest** (not continuous compounding), consistent
with precious metals market convention.

F\_OTC should be very close to the COMEX futures price in normal markets.
Any gap is the EFP.

In [ ]:
def compute_otc_forward(spot, fwd_rate_dec, delta_T):
    """Compute OTC forward price using simple interest.

    F_OTC = spot * (1 + fwd_rate * delta_T)

    Parameters
    ----------
    spot : float
        Current spot price.
    fwd_rate_dec : float
        Forward rate as a decimal (e.g. 0.045 for 4.5%).
    delta_T : float
        Time to FND in years.

    Returns
    -------
    float
        OTC forward price.
    """
    if pd.isna(spot) or pd.isna(fwd_rate_dec) or pd.isna(delta_T):
        return np.nan
    if delta_T <= 0:
        return spot  # at FND, forward = spot
    return spot * (1.0 + fwd_rate_dec * delta_T)


# ── Apply ─────────────────────────────────────────────────────────
master_df['F_OTC'] = master_df.apply(
    lambda row: compute_otc_forward(
        row['spot'], row['fwd_rate_interp'], row['delta_T']
    ), axis=1
)

# ── Sanity check: F_OTC vs futures_price ─────────────────────────
print("=" * 70)
print("  OTC FORWARD vs COMEX FUTURES — SANITY CHECK")
print("=" * 70)

for metal in config['metals']:
    sub = master_df[master_df['metal'] == metal].dropna(
        subset=['futures_price', 'F_OTC'])
    if len(sub) == 0:
        continue

    diff = sub['futures_price'] - sub['F_OTC']
    print(f"\n  {metal.upper()} (futures_price - F_OTC):")
    print(f"    Mean   : ${diff.mean():+.4f}")
    print(f"    Std    : ${diff.std():.4f}")
    print(f"    Median : ${diff.median():+.4f}")
    print(f"    Range  : [${diff.min():+.4f}, ${diff.max():+.4f}]")
    print(f"    Latest : ${diff.iloc[-1]:+.4f}  "
          f"(futures=${sub['futures_price'].iloc[-1]:,.2f}, "
          f"F_OTC=${sub['F_OTC'].iloc[-1]:,.2f})")

### F\_OTC vs Futures Price — Visual Check

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for i, metal in enumerate(config['metals']):
    ax = axes[i]
    sub = master_df[master_df['metal'] == metal].sort_values('date')

    ax.plot(sub['date'], sub['futures_price'],
            linewidth=1.2, color='#2c3e50', label='COMEX Futures')
    ax.plot(sub['date'], sub['F_OTC'],
            linewidth=1.2, color='#e74c3c', linestyle='--', label='OTC Forward')

    ax.set_title(f'{metal.upper()} — COMEX Futures vs OTC Forward (anchored to FND)',
                 fontsize=12)
    ax.set_ylabel('Price ($/oz)')
    ax.legend(loc='best', fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Section 10 — EFP Series Construction

Six EFP measures computed from master\_df:

| Column | Definition | Interpretation |
|--------|-----------|----------------|
| `EFP_raw` | futures − spot | Classic basis; includes carry |
| `EFP_adj` | futures − F\_OTC | Adjusted EFP; strips carry, isolates COMEX vs OTC basis |
| `EFP_raw_pct` | (EFP\_raw / spot) × 100 | Basis as % of spot |
| `EFP_adj_pct` | (EFP\_adj / spot) × 100 | Adjusted basis as % |
| `EFP_raw_ann` | EFP\_raw / (spot × delta\_T) | Annualised raw basis |
| `EFP_adj_ann` | EFP\_adj / (spot × delta\_T) | Annualised adjusted basis |

`EFP_adj` close to zero = normal market. Spikes indicate physical
scarcity, tariff risk, or cross-market dislocation.

In [ ]:
# ── EFP computations ──────────────────────────────────────────────
# 1. Raw EFP (classic basis)
master_df['EFP_raw'] = master_df['futures_price'] - master_df['spot']

# 2. Adjusted EFP (carry-stripped)
master_df['EFP_adj'] = master_df['futures_price'] - master_df['F_OTC']

# 3. Percentage versions
master_df['EFP_raw_pct'] = (master_df['EFP_raw'] / master_df['spot']) * 100
master_df['EFP_adj_pct'] = (master_df['EFP_adj'] / master_df['spot']) * 100

# 4. Annualised versions (guard against tiny delta_T)
min_delta_T_for_ann = 0.01  # ~3.65 days

master_df['EFP_adj_ann'] = np.where(
    master_df['delta_T'] >= min_delta_T_for_ann,
    master_df['EFP_adj'] / (master_df['spot'] * master_df['delta_T']),
    np.nan
)

master_df['EFP_raw_ann'] = np.where(
    master_df['delta_T'] >= min_delta_T_for_ann,
    master_df['EFP_raw'] / (master_df['spot'] * master_df['delta_T']),
    np.nan
)

# ── Descriptive statistics ───────────────────────────────────────
print("=" * 75)
print("  EFP DESCRIPTIVE STATISTICS")
print("=" * 75)

for metal in config['metals']:
    sub = master_df[master_df['metal'] == metal]
    print(f"\n  {metal.upper()}:")

    for col, unit in [('EFP_raw', '$/oz'), ('EFP_adj', '$/oz'),
                      ('EFP_raw_pct', '%'), ('EFP_adj_pct', '%'),
                      ('EFP_adj_ann', 'ann dec')]:
        s = sub[col].dropna()
        if len(s) == 0:
            continue
        print(f"\n    {col} ({unit}):")
        print(f"      Mean   : {s.mean():+.4f}")
        print(f"      Std    : {s.std():.4f}")
        print(f"      Median : {s.median():+.4f}")
        print(f"      Min    : {s.min():+.4f}")
        print(f"      Max    : {s.max():+.4f}")
        if 'adj' in col.lower() and 'ann' not in col.lower():
            pct_pos = (s > 0).mean() * 100
            print(f"      %% > 0  : {pct_pos:.1f}%")

# ── Data quality flag: exclude_from_regression ───────────────────
# Rolling stats for outlier detection
for metal in config['metals']:
    mask = master_df['metal'] == metal
    roll_mean = master_df.loc[mask, 'EFP_adj'].rolling(60, min_periods=20).mean()
    roll_std  = master_df.loc[mask, 'EFP_adj'].rolling(60, min_periods=20).std()
    master_df.loc[mask, '_efp_adj_zscore'] = (
        (master_df.loc[mask, 'EFP_adj'] - roll_mean) / roll_std.replace(0, np.nan)
    ).abs()

master_df['exclude_from_regression'] = (
    (master_df['is_roll_date'] == True) |
    (master_df['delta_T'] < 5/365) |
    (master_df['fwd_rate_interp'].isna()) |
    (master_df['_efp_adj_zscore'] > 10)
)

# Drop temp column
master_df = master_df.drop(columns=['_efp_adj_zscore'], errors='ignore')

n_excluded = master_df['exclude_from_regression'].sum()
print(f"\n  Exclusion flag summary:")
print(f"    Total rows           : {len(master_df):,}")
print(f"    Excluded             : {n_excluded:,} ({n_excluded/len(master_df)*100:.1f}%)")
print(f"    Available for regr.  : {len(master_df) - n_excluded:,}")
print(f"    Breakdown:")
print(f"      Roll dates         : {master_df['is_roll_date'].sum()}")
print(f"      Near expiry (<5d)  : {(master_df['delta_T'] < 5/365).sum()}")
print(f"      NaN fwd rate       : {master_df['fwd_rate_interp'].isna().sum()}")

## Section 11 — Daily Differenced Series for Regression

Compute first-differences for the regression target and regressors:
- `delta_spot` = daily change in spot
- `delta_efp_raw` = daily change in raw EFP (basis)
- `delta_efp_adj` = daily change in adjusted EFP

Differences are set to NaN on roll dates and exclusion-flagged dates
to prevent spurious jumps from contaminating the regression.

In [ ]:
# ── Compute daily differences per metal ───────────────────────────
for metal in config['metals']:
    mask = master_df['metal'] == metal
    idx = master_df.loc[mask].index

    master_df.loc[idx, 'delta_spot']    = master_df.loc[idx, 'spot'].diff()
    master_df.loc[idx, 'delta_efp_raw'] = master_df.loc[idx, 'EFP_raw'].diff()
    master_df.loc[idx, 'delta_efp_adj'] = master_df.loc[idx, 'EFP_adj'].diff()

# ── Null out differences on excluded dates ───────────────────────
exclude_mask = master_df['exclude_from_regression']
master_df.loc[exclude_mask, ['delta_spot', 'delta_efp_raw', 'delta_efp_adj']] = np.nan

# Also null out the day AFTER a roll (the diff would span the roll)
for metal in config['metals']:
    mask = master_df['metal'] == metal
    roll_idx = master_df.loc[mask & master_df['is_roll_date']].index
    # The next business day after each roll also has a tainted diff
    for ri in roll_idx:
        pos = master_df.index.get_loc(ri)
        if pos + 1 < len(master_df) and master_df.iloc[pos + 1]['metal'] == metal:
            next_idx = master_df.index[pos + 1]
            master_df.loc[next_idx, ['delta_spot', 'delta_efp_raw', 'delta_efp_adj']] = np.nan

# ── Summary ──────────────────────────────────────────────────────
print("=" * 70)
print("  DAILY DIFFERENCED SERIES SUMMARY")
print("=" * 70)

for metal in config['metals']:
    sub = master_df[master_df['metal'] == metal]
    print(f"\n  {metal.upper()}:")
    for col in ['delta_spot', 'delta_efp_raw', 'delta_efp_adj']:
        s = sub[col].dropna()
        print(f"    {col:18s}  n={len(s):>5d}  mean={s.mean():+.4f}  "
              f"std={s.std():.4f}  range=[{s.min():+.4f}, {s.max():+.4f}]")

    # Correlation check
    clean = sub[['delta_spot', 'delta_efp_raw', 'delta_efp_adj']].dropna()
    if len(clean) > 20:
        corr_raw = clean['delta_spot'].corr(clean['delta_efp_raw'])
        corr_adj = clean['delta_spot'].corr(clean['delta_efp_adj'])
        print(f"    corr(delta_spot, delta_efp_raw) = {corr_raw:.4f}")
        print(f"    corr(delta_spot, delta_efp_adj) = {corr_adj:.4f}")

## Section 12 — EFP Time Series Visualisations

Four-panel view:
1. Gold: EFP\_raw and EFP\_adj ($/oz)
2. Silver: EFP\_raw and EFP\_adj ($/oz)
3. Gold: EFP\_adj\_ann (annualised adjusted EFP)
4. Silver: EFP\_adj\_ann

Roll dates shaded in light grey. Major EFP events annotated.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col_idx, metal in enumerate(config['metals']):
    sub = master_df[master_df['metal'] == metal].sort_values('date').copy()

    # ── Top row: EFP_raw and EFP_adj ─────────────────────────────
    ax = axes[0, col_idx]
    ax.plot(sub['date'], sub['EFP_raw'],
            linewidth=0.9, color='#7f8c8d', alpha=0.7, label='EFP_raw (F-S)')
    ax.plot(sub['date'], sub['EFP_adj'],
            linewidth=1.2, color='#2c3e50', label='EFP_adj (F-F_OTC)')
    ax.axhline(y=0, color='red', linestyle='--', linewidth=0.5, alpha=0.5)

    # Shade roll dates
    roll_dates = sub[sub['is_roll_date']]['date']
    for rd in roll_dates:
        ax.axvline(x=rd, color='lightgrey', linewidth=0.3, alpha=0.5)

    # Annotate tariff period (Jan-Feb 2025) if in range
    tariff_date = pd.Timestamp('2025-01-15')
    if sub['date'].min() <= tariff_date <= sub['date'].max():
        ax.axvline(x=tariff_date, color='#e74c3c', linestyle='--',
                   linewidth=1, alpha=0.7)
        y_pos = sub['EFP_adj'].max() * 0.85
        ax.annotate('Tariff risk\nJan 2025', xy=(tariff_date, y_pos),
                    fontsize=8, color='#e74c3c', ha='right',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                              edgecolor='#e74c3c', alpha=0.8))

    unit = '$/oz'
    ax.set_title(f'{metal.upper()} — EFP Raw vs Adjusted ({unit})', fontsize=11)
    ax.set_ylabel(f'EFP ({unit})')
    ax.legend(loc='best', fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.sca(ax)
    plt.xticks(rotation=45, fontsize=8)

    # ── Bottom row: EFP_adj_ann ──────────────────────────────────
    ax2 = axes[1, col_idx]
    ann = sub['EFP_adj_ann'].dropna()
    ax2.plot(sub['date'], sub['EFP_adj_ann'] * 100,
             linewidth=1.0, color='#8e44ad')
    ax2.axhline(y=0, color='red', linestyle='--', linewidth=0.5, alpha=0.5)

    for rd in roll_dates:
        ax2.axvline(x=rd, color='lightgrey', linewidth=0.3, alpha=0.5)

    if sub['date'].min() <= tariff_date <= sub['date'].max():
        ax2.axvline(x=tariff_date, color='#e74c3c', linestyle='--',
                    linewidth=1, alpha=0.7)

    ax2.set_title(f'{metal.upper()} — Annualised Adjusted EFP (%)', fontsize=11)
    ax2.set_ylabel('EFP_adj_ann (%)')
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.sca(ax2)
    plt.xticks(rotation=45, fontsize=8)

plt.tight_layout()
plt.show()

### EFP Summary Table

In [ ]:
# ── Summary table ─────────────────────────────────────────────────
print("=" * 80)
print("  EFP SUMMARY TABLE")
print("=" * 80)
print(f"\n  {'Metal':8s}  {'Mean Raw':>10s}  {'Mean Adj':>10s}  {'Std Adj':>10s}  "
      f"{'Max Adj':>10s}  {'Min Adj':>10s}  {'%>0 Adj':>8s}")
print(f"  {'-'*72}")

for metal in config['metals']:
    sub = master_df[master_df['metal'] == metal]
    raw = sub['EFP_raw'].dropna()
    adj = sub['EFP_adj'].dropna()
    pct_pos = (adj > 0).mean() * 100 if len(adj) > 0 else 0

    print(f"  {metal.upper():8s}  "
          f"${raw.mean():>+8.2f}  "
          f"${adj.mean():>+8.4f}  "
          f"${adj.std():>8.4f}  "
          f"${adj.max():>+8.4f}  "
          f"${adj.min():>+8.4f}  "
          f"{pct_pos:>6.1f}%")

# ── Latest values ────────────────────────────────────────────────
print(f"\n  Latest values:")
for metal in config['metals']:
    sub = master_df[master_df['metal'] == metal].sort_values('date')
    if len(sub) == 0:
        continue
    latest = sub.iloc[-1]
    print(f"\n  {metal.upper()} ({latest['date']:%Y-%m-%d}):")
    print(f"    Spot           : ${latest['spot']:>10,.2f}")
    print(f"    COMEX Futures  : ${latest['futures_price']:>10,.2f}")
    print(f"    OTC Forward    : ${latest['F_OTC']:>10,.2f}")
    print(f"    EFP_raw        : ${latest['EFP_raw']:>+10.2f}")
    print(f"    EFP_adj        : ${latest['EFP_adj']:>+10.4f}")
    if pd.notna(latest.get('EFP_adj_ann')):
        print(f"    EFP_adj_ann    : {latest['EFP_adj_ann']*100:>+10.4f}%")

### Export Updated Master DataFrame

In [ ]:
# ── Save master_df with EFP columns ──────────────────────────────
output_path = 'efp_with_spreads.csv'
master_df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"  {master_df.shape[0]:,} rows x {master_df.shape[1]} columns")
print(f"\nColumns: {list(master_df.columns)}")

# EFP columns added in Prompt B
efp_cols = ['fwd_rate_interp', 'F_OTC', 'EFP_raw', 'EFP_adj',
            'EFP_raw_pct', 'EFP_adj_pct', 'EFP_raw_ann', 'EFP_adj_ann',
            'exclude_from_regression', 'delta_spot', 'delta_efp_raw', 'delta_efp_adj']
print(f"\nNew columns from Prompt B ({len(efp_cols)}):")
for col in efp_cols:
    n_valid = master_df[col].notna().sum() if col in master_df.columns else 0
    print(f"  {col:28s}: {n_valid:>6,} valid values")

print(f"\nPrompt B complete: {datetime.now():%Y-%m-%d %H:%M}")
print("master_df is ready for regression in Prompt C.")

---
# Prompt C — OLS Regression, Rolling Beta, Theoretical Beta

> Sections 13–17: Static and rolling OLS regressions of EFP changes
> on spot changes, theoretical beta from cost-of-carry, regression
> diagnostics, and practical delta-exposure interpretation.

In [ ]:
# ── Reload guard: load from CSV if master_df not in memory ────────
try:
    _ = master_df.shape
    print(f"master_df already in memory: {master_df.shape}")
except NameError:
    import pandas as pd
    import numpy as np
    import statsmodels.api as sm
    from statsmodels.regression.rolling import RollingOLS
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    import seaborn as sns
    from scipy import stats as sp_stats
    from datetime import date, datetime
    import warnings
    warnings.filterwarnings("ignore")

    master_df = pd.read_csv('efp_with_spreads.csv', parse_dates=['date', 'fnd'])
    print(f"Loaded master_df from CSV: {master_df.shape}")

    # Rebuild config if not present
    config = {
        'metals': ['gold', 'silver'],
        'start_date': '2023-01-01',
        'end_date': date.today().strftime('%Y-%m-%d'),
        'roll_days_before_fnd': 5,
        'regression_window_days': 60,
        'k_sigma': 3,
        'generic_depth': 4,
    }

# Ensure imports available regardless
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
from scipy import stats as sp_stats

## Section 13 — Static OLS Regression: Empirical Beta

Full-sample OLS regression of daily EFP changes on daily spot changes:

$$\Delta \text{EFP\_adj}(t) = \alpha + \beta_{spot} \times \Delta \text{spot}(t) + \varepsilon(t)$$

**Newey-West HAC** standard errors (lags=5) correct for autocorrelation
in the residuals. A statistically significant positive $\beta_{spot}$
means the EFP widens when spot rises — i.e. COMEX richens relative to
OTC forwards when the market rallies.

Also runs a **level regression** (EFP\_adj on spot) to estimate how
many dollars the EFP level moves per dollar of spot, which should
approximate $r_{fwd} \times \Delta T$ theoretically.

In [ ]:
# ── Static OLS: changes regression ────────────────────────────────
static_results = {}

print("=" * 80)
print("  SECTION 13 — STATIC OLS: delta_EFP_adj ~ delta_spot")
print("=" * 80)

for metal in config['metals']:
    sub = master_df[
        (master_df['metal'] == metal) &
        (~master_df['exclude_from_regression'])
    ][['date', 'delta_spot', 'delta_efp_adj']].dropna()

    if len(sub) < 30:
        print(f"\n  {metal.upper()}: insufficient data ({len(sub)} obs)")
        continue

    y = sub['delta_efp_adj'].values
    X = sm.add_constant(sub['delta_spot'].values)

    model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 5})

    static_results[metal] = {
        'model': model,
        'n_obs': len(sub),
        'date_range': (sub['date'].min(), sub['date'].max()),
        'alpha': model.params[0],
        'beta_spot': model.params[1],
        'se_beta': model.bse[1],
        't_stat': model.tvalues[1],
        'p_value': model.pvalues[1],
        'r_squared': model.rsquared,
    }

    r = static_results[metal]
    sig = "***" if r['p_value'] < 0.001 else "**" if r['p_value'] < 0.01 \
          else "*" if r['p_value'] < 0.05 else ""

    print(f"\n  {metal.upper()} — Changes Regression")
    print(f"  {'─' * 55}")
    print(f"  delta_EFP_adj = alpha + beta_spot × delta_spot + eps")
    print(f"  {'─' * 55}")
    print(f"    beta_spot    : {r['beta_spot']:+.6f} {sig}")
    print(f"    Std error    : {r['se_beta']:.6f}  (HAC, lags=5)")
    print(f"    t-statistic  : {r['t_stat']:+.3f}")
    print(f"    p-value      : {r['p_value']:.4e}")
    print(f"    alpha        : {r['alpha']:+.6f}")
    print(f"    R-squared    : {r['r_squared']:.4f}")
    print(f"    Observations : {r['n_obs']:,}")
    print(f"    Sample       : {r['date_range'][0]:%Y-%m-%d} to "
          f"{r['date_range'][1]:%Y-%m-%d}")

# ── Level regression: EFP_adj ~ spot ─────────────────────────────
print("\n" + "=" * 80)
print("  STATIC OLS: EFP_adj_level ~ spot_level")
print("=" * 80)

level_results = {}

for metal in config['metals']:
    sub = master_df[
        (master_df['metal'] == metal) &
        (~master_df['exclude_from_regression'])
    ][['date', 'spot', 'EFP_adj']].dropna()

    if len(sub) < 30:
        continue

    y = sub['EFP_adj'].values
    X = sm.add_constant(sub['spot'].values)

    model_lvl = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 10})

    level_results[metal] = {
        'beta_level': model_lvl.params[1],
        'se': model_lvl.bse[1],
        't_stat': model_lvl.tvalues[1],
        'p_value': model_lvl.pvalues[1],
        'r_squared': model_lvl.rsquared,
    }

    lr = level_results[metal]
    sig = "***" if lr['p_value'] < 0.001 else "**" if lr['p_value'] < 0.01 \
          else "*" if lr['p_value'] < 0.05 else ""

    # Theoretical comparison
    avg_fwd = master_df.loc[
        (master_df['metal'] == metal) &
        (~master_df['exclude_from_regression']),
        'fwd_rate_interp'
    ].mean()
    avg_dT = master_df.loc[
        (master_df['metal'] == metal) &
        (~master_df['exclude_from_regression']),
        'delta_T'
    ].mean()

    print(f"\n  {metal.upper()} — Level Regression")
    print(f"  {'─' * 55}")
    print(f"    beta_level   : {lr['beta_level']:+.6f} {sig}")
    print(f"    Std error    : {lr['se']:.6f}  (HAC, lags=10)")
    print(f"    t-statistic  : {lr['t_stat']:+.3f}")
    print(f"    p-value      : {lr['p_value']:.4e}")
    print(f"    R-squared    : {lr['r_squared']:.4f}")
    print(f"    Theoretical  : fwd_rate × delta_T ≈ {avg_fwd:.4f} × {avg_dT:.4f} "
          f"= {avg_fwd * avg_dT:.6f}")

### Interpretation of Static Beta

**Changes regression** ($\Delta$EFP\_adj ~ $\Delta$spot):
- $\beta_{spot}$ measures: for every \$1 that spot moves, by how many dollars
  does the carry-stripped EFP change *on the same day*?
- A positive, significant $\beta_{spot}$ implies the COMEX futures richens (cheapens)
  versus OTC forwards when spot rallies (sells off). This is the **residual
  directional delta** embedded in an EFP position beyond pure carry.
- If $\beta_{spot} \approx 0$, the EFP is purely a carry/financing trade with
  no residual spot exposure.

**Level regression** (EFP\_adj ~ spot):
- $\beta_{level}$ estimates how many dollars the EFP level changes per
  dollar of spot. Theoretically this should be close to $r_{fwd} \times \Delta T$
  (the carry component's sensitivity to spot).
- In practice, the level regression captures both carry sensitivity *and*
  any persistent co-movement between the EFP and spot (e.g. during tariff periods).

## Section 14 — Rolling OLS: Time-Varying Beta

A 60-day (configurable) rolling window OLS captures how the EFP's
sensitivity to spot evolves over time. Key questions:

- Does $\beta_{spot}$ spike during stress episodes (tariffs, physical scarcity)?
- Is the beta stable enough to hedge, or does it require dynamic adjustment?
- Does the 95% confidence band exclude zero consistently?

In [ ]:
# ── Rolling OLS per metal ─────────────────────────────────────────
window = config['regression_window_days']
rolling_frames = []

print("=" * 80)
print(f"  SECTION 14 — ROLLING OLS (window={window} days)")
print("=" * 80)

for metal in config['metals']:
    sub = master_df[
        (master_df['metal'] == metal) &
        (~master_df['exclude_from_regression'])
    ][['date', 'delta_spot', 'delta_efp_adj']].dropna().copy()

    sub = sub.sort_values('date').reset_index(drop=True)

    if len(sub) < window + 10:
        print(f"  {metal.upper()}: insufficient data ({len(sub)} obs for window={window})")
        continue

    y = sub['delta_efp_adj']
    X = sm.add_constant(sub['delta_spot'])

    rols = RollingOLS(y, X, window=window).fit()

    sub['rolling_beta']  = rols.params.iloc[:, 1].values
    sub['rolling_alpha'] = rols.params.iloc[:, 0].values
    sub['rolling_rsq']   = rols.rsquared.values
    sub['rolling_se']    = rols.bse.iloc[:, 1].values

    # 95% confidence bands
    sub['upper_95'] = sub['rolling_beta'] + 1.96 * sub['rolling_se']
    sub['lower_95'] = sub['rolling_beta'] - 1.96 * sub['rolling_se']

    sub['metal'] = metal

    rolling_frames.append(sub[[
        'date', 'metal', 'rolling_beta', 'rolling_alpha',
        'rolling_rsq', 'rolling_se', 'upper_95', 'lower_95'
    ]])

    valid = sub['rolling_beta'].dropna()
    print(f"\n  {metal.upper()}:")
    print(f"    Rolling beta  : mean={valid.mean():+.6f}  "
          f"std={valid.std():.6f}  range=[{valid.min():+.6f}, {valid.max():+.6f}]")
    print(f"    Rolling R²    : mean={sub['rolling_rsq'].dropna().mean():.4f}")
    print(f"    % of windows with beta > 0  : "
          f"{(valid > 0).mean()*100:.1f}%")
    print(f"    % of windows where 95% CI excludes 0 : "
          f"{((sub['lower_95'].dropna() > 0) | (sub['upper_95'].dropna() < 0)).mean()*100:.1f}%")

rolling_betas = pd.concat(rolling_frames, ignore_index=True)
print(f"\nrolling_betas shape: {rolling_betas.shape}")

### Rolling Beta Visualisation

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

for i, metal in enumerate(config['metals']):
    ax = axes[i]
    sub = rolling_betas[rolling_betas['metal'] == metal].sort_values('date')

    ax.plot(sub['date'], sub['rolling_beta'],
            linewidth=1.2, color='#2c3e50', label='Rolling beta (60d)')
    ax.fill_between(sub['date'], sub['lower_95'], sub['upper_95'],
                    alpha=0.2, color='#3498db', label='95% CI')
    ax.axhline(y=0, color='red', linestyle='--', linewidth=0.8, alpha=0.6)

    # Static beta reference line
    if metal in static_results:
        ax.axhline(y=static_results[metal]['beta_spot'],
                   color='#e67e22', linestyle=':', linewidth=1,
                   label=f"Static beta = {static_results[metal]['beta_spot']:.4f}")

    # Annotate tariff period
    tariff_date = pd.Timestamp('2025-01-15')
    if sub['date'].min() <= tariff_date <= sub['date'].max():
        ax.axvline(x=tariff_date, color='#e74c3c', linestyle='--',
                   linewidth=1, alpha=0.7)
        ax.annotate('Tariff risk\nJan 2025',
                    xy=(tariff_date, sub['rolling_beta'].max() * 0.9),
                    fontsize=8, color='#e74c3c', ha='right',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                              edgecolor='#e74c3c', alpha=0.8))

    ax.set_title(f'{metal.upper()} — Rolling Beta (delta_EFP_adj ~ delta_spot, '
                 f'{config["regression_window_days"]}d window)', fontsize=11)
    ax.set_ylabel('Beta')
    ax.legend(loc='best', fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Section 15 — Theoretical Beta from Cost-of-Carry

The cost-of-carry model implies:

$$\text{EFP\_adj} \approx S \times r_{fwd} \times \Delta T - S \times r_{fwd} \times \Delta T = 0$$

But for changes: $\frac{\partial \text{EFP\_adj}}{\partial S} \approx r_{fwd} \times \Delta T$

So the **theoretical beta** is:

$$\beta_{theo}(t) = r_{fwd,interp}(t) \times \Delta T(t)$$

Comparing rolling empirical beta to $\beta_{theo}$ reveals:
- **beta\_excess > 0**: COMEX richening faster than carry implies — physical scarcity / tariff premium
- **beta\_excess < 0**: COMEX cheapening — liquidity normalising or physical surplus

In [ ]:
# ── Theoretical beta ──────────────────────────────────────────────
master_df['beta_theo'] = master_df['fwd_rate_interp'] * master_df['delta_T']

print("=" * 80)
print("  SECTION 15 — THEORETICAL BETA (fwd_rate × delta_T)")
print("=" * 80)

for metal in config['metals']:
    sub = master_df[
        (master_df['metal'] == metal) &
        (~master_df['exclude_from_regression'])
    ]['beta_theo'].dropna()
    print(f"\n  {metal.upper()}:")
    print(f"    Mean   : {sub.mean():.6f}")
    print(f"    Median : {sub.median():.6f}")
    print(f"    Std    : {sub.std():.6f}")
    print(f"    Range  : [{sub.min():.6f}, {sub.max():.6f}]")

# ── Build beta_comparison DataFrame ──────────────────────────────
# Merge rolling betas with theoretical beta from master_df
beta_comparison_frames = []

for metal in config['metals']:
    rb = rolling_betas[rolling_betas['metal'] == metal][
        ['date', 'metal', 'rolling_beta', 'rolling_rsq',
         'upper_95', 'lower_95']
    ].copy()

    theo = master_df[master_df['metal'] == metal][
        ['date', 'beta_theo']
    ].copy()

    merged = rb.merge(theo, on='date', how='left')
    merged['beta_excess'] = merged['rolling_beta'] - merged['beta_theo']
    beta_comparison_frames.append(merged)

beta_comparison = pd.concat(beta_comparison_frames, ignore_index=True)

# Summary
print("\n" + "=" * 80)
print("  BETA COMPARISON: EMPIRICAL vs THEORETICAL")
print("=" * 80)

for metal in config['metals']:
    sub = beta_comparison[beta_comparison['metal'] == metal].dropna(
        subset=['rolling_beta', 'beta_theo'])
    if len(sub) == 0:
        continue
    print(f"\n  {metal.upper()}:")
    print(f"    Empirical beta  : mean={sub['rolling_beta'].mean():+.6f}")
    print(f"    Theoretical beta: mean={sub['beta_theo'].mean():.6f}")
    print(f"    Beta excess     : mean={sub['beta_excess'].mean():+.6f}  "
          f"std={sub['beta_excess'].std():.6f}")
    print(f"    % excess > 0   : {(sub['beta_excess'] > 0).mean()*100:.1f}%")

print(f"\nbeta_comparison shape: {beta_comparison.shape}")

### Empirical vs Theoretical Beta — Visual Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col_idx, metal in enumerate(config['metals']):
    sub = beta_comparison[beta_comparison['metal'] == metal].sort_values('date')

    # ── Top row: empirical vs theoretical beta ────────────────────
    ax = axes[0, col_idx]
    ax.plot(sub['date'], sub['rolling_beta'],
            linewidth=1.2, color='#2c3e50', label='Empirical (rolling 60d)')
    ax.plot(sub['date'], sub['beta_theo'],
            linewidth=1.0, color='#27ae60', linestyle='--',
            label='Theoretical (fwd_rate × delta_T)')
    ax.axhline(y=0, color='red', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.set_title(f'{metal.upper()} — Empirical vs Theoretical Beta', fontsize=11)
    ax.set_ylabel('Beta')
    ax.legend(loc='best', fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.sca(ax)
    plt.xticks(rotation=45, fontsize=8)

    # ── Bottom row: beta excess ───────────────────────────────────
    ax2 = axes[1, col_idx]
    ax2.bar(sub['date'], sub['beta_excess'],
            width=1.5, color=np.where(sub['beta_excess'] > 0, '#e74c3c', '#3498db'),
            alpha=0.6)
    ax2.axhline(y=0, color='black', linewidth=0.8)
    ax2.set_title(f'{metal.upper()} — Beta Excess (Empirical − Theoretical)',
                  fontsize=11)
    ax2.set_ylabel('Beta Excess')
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.sca(ax2)
    plt.xticks(rotation=45, fontsize=8)

    # Annotate tariff period
    tariff_date = pd.Timestamp('2025-01-15')
    for a in [ax, ax2]:
        if sub['date'].min() <= tariff_date <= sub['date'].max():
            a.axvline(x=tariff_date, color='#e74c3c', linestyle='--',
                      linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.show()

## Section 16 — Regression Diagnostics

Diagnostic checks for the static OLS (changes regression):
1. Residuals time series — check for autocorrelation patterns
2. Q-Q plot — normality of residuals
3. Histogram with normal overlay — fat tails check
4. Scatter of delta\_spot vs delta\_efp\_adj with OLS fit line

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 18))

for col_idx, metal in enumerate(config['metals']):
    if metal not in static_results:
        continue

    model = static_results[metal]['model']
    resid = model.resid

    # Dates for the regression sample
    sub = master_df[
        (master_df['metal'] == metal) &
        (~master_df['exclude_from_regression'])
    ][['date', 'delta_spot', 'delta_efp_adj']].dropna()
    dates = sub['date'].values

    # ── Row 0: Residuals time series ──────────────────────────────
    ax = axes[0, col_idx]
    ax.plot(dates, resid, linewidth=0.6, color='#2c3e50', alpha=0.7)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=0.5)
    ax.set_title(f'{metal.upper()} — OLS Residuals', fontsize=11)
    ax.set_ylabel('Residual')

    # Shade tariff period
    tariff_start = pd.Timestamp('2025-01-01')
    tariff_end = pd.Timestamp('2025-03-01')
    ax.axvspan(tariff_start, tariff_end, alpha=0.15, color='red',
               label='Tariff period')
    ax.legend(loc='best', fontsize=8)

    # ── Row 1: Q-Q plot ───────────────────────────────────────────
    ax = axes[1, col_idx]
    sp_stats.probplot(resid, dist="norm", plot=ax)
    ax.set_title(f'{metal.upper()} — Q-Q Plot of Residuals', fontsize=11)
    ax.get_lines()[0].set_markersize(3)

    # ── Row 2: Histogram ──────────────────────────────────────────
    ax = axes[2, col_idx]
    ax.hist(resid, bins=60, density=True, alpha=0.7, color='#3498db',
            edgecolor='white', linewidth=0.5)
    # Normal overlay
    x_range = np.linspace(resid.min(), resid.max(), 200)
    ax.plot(x_range, sp_stats.norm.pdf(x_range, resid.mean(), resid.std()),
            color='#e74c3c', linewidth=1.5, label='Normal')
    ax.set_title(f'{metal.upper()} — Residual Histogram', fontsize=11)
    ax.set_xlabel('Residual')
    ax.set_ylabel('Density')
    ax.legend(loc='best', fontsize=8)

    # ── Row 3: Scatter with OLS line ──────────────────────────────
    ax = axes[3, col_idx]
    ax.scatter(sub['delta_spot'].values, sub['delta_efp_adj'].values,
               s=8, alpha=0.4, color='#3498db')
    # OLS line
    x_line = np.linspace(sub['delta_spot'].min(), sub['delta_spot'].max(), 100)
    y_line = model.params[0] + model.params[1] * x_line
    ax.plot(x_line, y_line, color='#e74c3c', linewidth=1.5,
            label=f'OLS: beta={model.params[1]:.4f}')
    ax.axhline(y=0, color='grey', linestyle='--', linewidth=0.3)
    ax.axvline(x=0, color='grey', linestyle='--', linewidth=0.3)
    ax.set_title(f'{metal.upper()} — delta_spot vs delta_EFP_adj', fontsize=11)
    ax.set_xlabel('delta_spot ($/oz)')
    ax.set_ylabel('delta_EFP_adj ($/oz)')
    ax.legend(loc='best', fontsize=8)

plt.tight_layout()
plt.show()

### Diagnostic Statistics

In [ ]:
from statsmodels.stats.stattools import durbin_watson

print("=" * 80)
print("  REGRESSION DIAGNOSTIC STATISTICS")
print("=" * 80)

for metal in config['metals']:
    if metal not in static_results:
        continue

    model = static_results[metal]['model']
    resid = model.resid

    dw = durbin_watson(resid)
    kurt = sp_stats.kurtosis(resid, fisher=True)  # excess kurtosis
    skew = sp_stats.skew(resid)
    jb_stat, jb_p = sp_stats.jarque_bera(resid)

    print(f"\n  {metal.upper()}:")
    print(f"    Durbin-Watson         : {dw:.4f}  "
          f"({'≈2 OK' if 1.5 < dw < 2.5 else 'autocorrelation detected'})")
    print(f"    Excess kurtosis       : {kurt:.2f}  "
          f"({'fat tails' if abs(kurt) > 1 else 'near-normal'})")
    print(f"    Skewness              : {skew:+.2f}")
    print(f"    Jarque-Bera statistic : {jb_stat:.1f}  (p={jb_p:.4e})")
    print(f"    Residual std          : {resid.std():.6f}")

    # Check for structural breaks: compare first/second half betas
    sub = master_df[
        (master_df['metal'] == metal) &
        (~master_df['exclude_from_regression'])
    ][['date', 'delta_spot', 'delta_efp_adj']].dropna()

    mid = len(sub) // 2
    for label, half in [('First half', sub.iloc[:mid]), ('Second half', sub.iloc[mid:])]:
        y_h = half['delta_efp_adj'].values
        X_h = sm.add_constant(half['delta_spot'].values)
        m_h = sm.OLS(y_h, X_h).fit()
        print(f"    {label:12s} beta : {m_h.params[1]:+.6f}  "
              f"(n={len(half)}, R²={m_h.rsquared:.4f})")

### Diagnostic Notes

- **Durbin-Watson ≈ 2** → no strong first-order autocorrelation in residuals
  (HAC standard errors already guard against this, but DW confirms).
- **Excess kurtosis > 0** → fat tails are expected in financial data and
  confirmed by Q-Q plots. This is why we use HAC and interpret p-values cautiously.
- **Structural break check**: comparing first-half vs second-half betas
  detects regime shifts. A large difference (especially around the 2025 tariff
  period) suggests the relationship is non-stationary and supports using
  rolling rather than static betas for risk management.

## Section 17 — Practical Interpretation: Delta Exposure

For a trader holding an EFP position (long COMEX futures / short OTC forward),
the residual directional P&L from a spot move is:

$$\text{USD delta} = \text{position\_oz} \times \beta \times \frac{S}{100}$$

where $S/100$ converts a 1% spot move to dollars.

This answers: *"If I am long X oz of EFP and spot moves 1%, how much
residual P&L do I make/lose beyond the carry?"*

In [ ]:
def compute_delta_exposure(position_oz, beta, spot_price):
    """Compute USD delta exposure for a 1% spot move.

    Parameters
    ----------
    position_oz : float
        EFP position in troy ounces.
    beta : float
        Empirical or theoretical beta.
    spot_price : float
        Current spot price ($/oz).

    Returns
    -------
    float
        USD P&L for a 1% spot move.
    """
    return position_oz * beta * (spot_price / 100.0)


# ── Build scenario table ─────────────────────────────────────────
print("=" * 80)
print("  SECTION 17 — DELTA EXPOSURE TABLE")
print("=" * 80)

# Get latest values
scenarios = []

for metal in config['metals']:
    latest = master_df[master_df['metal'] == metal].sort_values('date').iloc[-1]
    spot = latest['spot']

    # Latest rolling beta
    rb_metal = rolling_betas[
        rolling_betas['metal'] == metal
    ].sort_values('date').dropna(subset=['rolling_beta'])
    emp_beta = rb_metal['rolling_beta'].iloc[-1] if len(rb_metal) > 0 else np.nan

    # Theoretical beta
    theo_beta = latest.get('beta_theo', np.nan)

    if metal == 'gold':
        positions = [1_000, 5_000, 10_000]
    else:
        positions = [5_000, 25_000, 50_000]

    for pos in positions:
        usd_emp = compute_delta_exposure(pos, emp_beta, spot)
        usd_theo = compute_delta_exposure(pos, theo_beta, spot)
        scenarios.append({
            'metal': metal.upper(),
            'position_oz': f"{pos:,}",
            'spot': f"${spot:,.2f}",
            'empirical_beta': f"{emp_beta:.6f}" if pd.notna(emp_beta) else "N/A",
            'theo_beta': f"{theo_beta:.6f}" if pd.notna(theo_beta) else "N/A",
            'USD_delta_empirical': f"${usd_emp:+,.2f}" if pd.notna(usd_emp) else "N/A",
            'USD_delta_theoretical': f"${usd_theo:+,.2f}" if pd.notna(usd_theo) else "N/A",
            'comment': 'Residual directional exposure over and above carry',
        })

scenario_df = pd.DataFrame(scenarios)

# Print formatted
print(f"\n  {'Metal':8s}  {'Position':>10s}  {'Spot':>10s}  {'Emp Beta':>12s}  "
      f"{'Theo Beta':>12s}  {'USD/1% (emp)':>14s}  {'USD/1% (theo)':>14s}")
print(f"  {'─' * 90}")

for _, row in scenario_df.iterrows():
    print(f"  {row['metal']:8s}  {row['position_oz']:>10s}  {row['spot']:>10s}  "
          f"{row['empirical_beta']:>12s}  {row['theo_beta']:>12s}  "
          f"{row['USD_delta_empirical']:>14s}  {row['USD_delta_theoretical']:>14s}")

print(f"\n  Comment: {scenarios[0]['comment']}")
print(f"\n  Note: USD delta = position_oz × beta × (spot / 100)")
print(f"  This is the P&L from a 1% spot move BEYOND carry.")

### Export Beta Comparison

In [ ]:
# ── Save beta_comparison ──────────────────────────────────────────
output_path = 'efp_beta_results.csv'
beta_comparison.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"  {beta_comparison.shape[0]:,} rows x {beta_comparison.shape[1]} columns")
print(f"  Columns: {list(beta_comparison.columns)}")

# ── Final summary ────────────────────────────────────────────────
print("\n" + "=" * 80)
print("  PROMPT C COMPLETE — SUMMARY")
print("=" * 80)

for metal in config['metals']:
    print(f"\n  {metal.upper()}:")

    if metal in static_results:
        sr = static_results[metal]
        print(f"    Static beta (changes)  : {sr['beta_spot']:+.6f}  "
              f"(p={sr['p_value']:.4e}, R²={sr['r_squared']:.4f})")

    rb_metal = rolling_betas[
        rolling_betas['metal'] == metal
    ].sort_values('date').dropna(subset=['rolling_beta'])
    if len(rb_metal) > 0:
        latest_rb = rb_metal.iloc[-1]
        print(f"    Latest rolling beta    : {latest_rb['rolling_beta']:+.6f}")
        print(f"    Latest rolling R²      : {latest_rb['rolling_rsq']:.4f}")

    bc = beta_comparison[beta_comparison['metal'] == metal].dropna(
        subset=['beta_excess'])
    if len(bc) > 0:
        latest_bc = bc.sort_values('date').iloc[-1]
        print(f"    Latest beta_theo       : {latest_bc['beta_theo']:.6f}")
        print(f"    Latest beta_excess     : {latest_bc['beta_excess']:+.6f}")

print(f"\nPrompt C complete: {datetime.now():%Y-%m-%d %H:%M}")
print("beta_comparison saved to 'efp_beta_results.csv'.")
print("Ready for Prompt D.")

---
# Prompt D — Visualisations, Dashboard, Sensitivity & Documentation

> Sections 18–23: Publication-quality charts, summary dashboard tables,
> sensitivity analysis, daily workflow guide, table of contents, and
> end-to-end validation on a known historical date.

In [ ]:
# ── Reload guard: load from CSV if DataFrames not in memory ───────
try:
    _ = master_df.shape
    _ = beta_comparison.shape
    _ = rolling_betas.shape
    _ = static_results
    print(f"All DataFrames in memory: master_df {master_df.shape}, "
          f"beta_comparison {beta_comparison.shape}, "
          f"rolling_betas {rolling_betas.shape}")
except NameError:
    import pandas as pd
    import numpy as np
    import statsmodels.api as sm
    from statsmodels.regression.rolling import RollingOLS
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    import matplotlib.ticker as mticker
    import seaborn as sns
    from scipy import stats as sp_stats
    from datetime import date, datetime
    import warnings
    warnings.filterwarnings("ignore")

    master_df = pd.read_csv('efp_with_spreads.csv', parse_dates=['date', 'fnd'])
    beta_comparison = pd.read_csv('efp_beta_results.csv', parse_dates=['date'])
    print(f"Loaded master_df: {master_df.shape}")
    print(f"Loaded beta_comparison: {beta_comparison.shape}")

    config = {
        'metals': ['gold', 'silver'],
        'start_date': '2023-01-01',
        'end_date': date.today().strftime('%Y-%m-%d'),
        'roll_days_before_fnd': 5,
        'regression_window_days': 60,
        'k_sigma': 3,
        'generic_depth': 4,
    }

    # Reconstruct rolling_betas from beta_comparison
    rolling_betas = beta_comparison[[
        'date', 'metal', 'rolling_beta', 'rolling_rsq',
        'upper_95', 'lower_95'
    ]].copy()

    # Reconstruct static_results by re-running static OLS
    static_results = {}
    for metal in config['metals']:
        sub = master_df[
            (master_df['metal'] == metal) &
            (~master_df['exclude_from_regression'].astype(bool))
        ][['date', 'delta_spot', 'delta_efp_adj', 'spot', 'EFP_adj']].dropna(
            subset=['delta_spot', 'delta_efp_adj'])
        if len(sub) < 30:
            continue
        y = sub['delta_efp_adj'].values
        X = sm.add_constant(sub['delta_spot'].values)
        model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
        static_results[metal] = {
            'model': model,
            'n_obs': len(sub),
            'date_range': (sub['date'].min(), sub['date'].max()),
            'alpha': model.params[0],
            'beta_spot': model.params[1],
            'se_beta': model.bse[1],
            't_stat': model.tvalues[1],
            'p_value': model.pvalues[1],
            'r_squared': model.rsquared,
        }

    # Level regression
    level_results = {}
    for metal in config['metals']:
        sub = master_df[
            (master_df['metal'] == metal) &
            (~master_df['exclude_from_regression'].astype(bool))
        ][['spot', 'EFP_adj']].dropna()
        if len(sub) < 30:
            continue
        y = sub['EFP_adj'].values
        X = sm.add_constant(sub['spot'].values)
        m = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 10})
        level_results[metal] = {
            'beta_level': m.params[1],
            'se': m.bse[1],
            't_stat': m.tvalues[1],
            'p_value': m.pvalues[1],
            'r_squared': m.rsquared,
        }

    # Ensure beta_theo exists
    if 'beta_theo' not in master_df.columns:
        master_df['beta_theo'] = (
            master_df['fwd_rate_interp'] * master_df['delta_T']
        )

    print("Static and level regressions rebuilt from CSV data.")

# Ensure matplotlib imports available
import matplotlib.ticker as mticker

## Section 18 — Core Visualisation Suite

Five publication-quality charts for buyside presentation:
1. Gold: rolling beta vs theoretical beta with 95% CI and COMEX-rich shading
2. Silver: same format
3. Gold: EFP\_adj with rolling beta overlay (dual axis)
4. Beta excess time series (both metals)
5. Delta exposure heatmaps (gold and silver)

### Charts 1 & 2 — Empirical vs Theoretical Beta (Gold & Silver)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

for i, metal in enumerate(config['metals']):
    ax = axes[i]
    bc = beta_comparison[beta_comparison['metal'] == metal].sort_values('date').copy()
    bc = bc.dropna(subset=['rolling_beta'])

    # Rolling beta line
    ax.plot(bc['date'], bc['rolling_beta'],
            linewidth=1.4, color='#2980b9', label='Empirical Beta (60d rolling)')

    # Theoretical beta line
    ax.plot(bc['date'], bc['beta_theo'],
            linewidth=1.2, color='#e67e22', linestyle='--',
            label='Theoretical Beta (fwd_rate × delta_T)')

    # 95% confidence band
    ax.fill_between(bc['date'], bc['lower_95'], bc['upper_95'],
                    alpha=0.15, color='#2980b9', label='95% Confidence Band')

    # Shade where beta_excess > 0 (COMEX rich)
    excess_pos = bc['beta_excess'] > 0
    if excess_pos.any():
        ax.fill_between(bc['date'],
                        ax.get_ylim()[0] if i == 0 else bc['rolling_beta'].min() * 1.5,
                        ax.get_ylim()[1] if i == 0 else bc['rolling_beta'].max() * 1.5,
                        where=excess_pos.values,
                        alpha=0.08, color='#e74c3c', label='COMEX Rich (excess > 0)')

    ax.axhline(y=0, color='grey', linestyle='--', linewidth=0.5)

    # Tariff annotation
    tariff_date = pd.Timestamp('2025-01-15')
    if len(bc) > 0 and bc['date'].min() <= tariff_date <= bc['date'].max():
        ax.axvline(x=tariff_date, color='#c0392b', linestyle='--',
                   linewidth=0.8, alpha=0.6)
        ax.annotate('Tariff\nRisk', xy=(tariff_date, bc['rolling_beta'].max() * 0.92),
                    fontsize=8, color='#c0392b', ha='right',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                              edgecolor='#c0392b', alpha=0.8))

    ax.set_title(f'{metal.upper()} EFP: Empirical vs Theoretical Beta to Spot',
                 fontsize=13, fontweight='bold')
    ax.set_ylabel('Beta (delta_EFP_adj / delta_spot)', fontsize=10)
    ax.legend(loc='best', fontsize=8, framealpha=0.9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.grid(axis='y', alpha=0.2)

# Re-apply COMEX-rich shading with correct y-limits after auto-scaling
for i, metal in enumerate(config['metals']):
    ax = axes[i]
    bc = beta_comparison[beta_comparison['metal'] == metal].sort_values('date').copy()
    bc = bc.dropna(subset=['rolling_beta'])
    excess_pos = bc['beta_excess'] > 0
    ylims = ax.get_ylim()
    if excess_pos.any():
        ax.fill_between(bc['date'], ylims[0], ylims[1],
                        where=excess_pos.values,
                        alpha=0.06, color='#e74c3c')
    ax.set_ylim(ylims)

plt.tight_layout()
plt.show()

### Chart 3 — Gold EFP Adjusted with Rolling Beta Overlay

In [ ]:
fig, ax1 = plt.subplots(figsize=(15, 6))

metal = 'gold'
sub = master_df[master_df['metal'] == metal].sort_values('date').copy()
rb = rolling_betas[rolling_betas['metal'] == metal].sort_values('date').copy()

# Left axis: EFP_adj
color_efp = '#2c3e50'
ax1.plot(sub['date'], sub['EFP_adj'], linewidth=1.0, color=color_efp,
         alpha=0.8, label='EFP_adj ($/oz)')
ax1.set_ylabel('EFP_adj ($/oz)', color=color_efp, fontsize=11)
ax1.tick_params(axis='y', labelcolor=color_efp)
ax1.axhline(y=0, color='grey', linestyle='--', linewidth=0.5)

# Annotate extreme EFP periods (> 2 std devs)
efp_mean = sub['EFP_adj'].mean()
efp_std = sub['EFP_adj'].std()
extreme_mask = (sub['EFP_adj'] > efp_mean + 2 * efp_std) | \
               (sub['EFP_adj'] < efp_mean - 2 * efp_std)
extreme_dates = sub.loc[extreme_mask, 'date']
for ed in extreme_dates:
    ax1.axvline(x=ed, color='#e74c3c', linewidth=0.3, alpha=0.3)

if len(extreme_dates) > 0:
    ax1.axvline(x=extreme_dates.iloc[0], color='#e74c3c', linewidth=0.3,
                alpha=0.3, label=f'Extreme EFP (>{efp_std*2:.2f} from mean)')

# Right axis: rolling_beta
ax2 = ax1.twinx()
color_beta = '#8e44ad'
rb_clean = rb.dropna(subset=['rolling_beta'])
ax2.plot(rb_clean['date'], rb_clean['rolling_beta'],
         linewidth=1.3, color=color_beta, alpha=0.8,
         label='Rolling Beta (60d)')
ax2.set_ylabel('Rolling Beta', color=color_beta, fontsize=11)
ax2.tick_params(axis='y', labelcolor=color_beta)

ax1.set_title('Gold EFP Adjusted vs Rolling Spot Beta', fontsize=13,
              fontweight='bold')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax1.tick_params(axis='x', rotation=45, labelsize=8)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

### Chart 4 — Beta Excess: Gold vs Silver

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))

for metal, color, lbl in [('gold', '#2980b9', 'Gold'),
                           ('silver', '#e67e22', 'Silver')]:
    bc = beta_comparison[beta_comparison['metal'] == metal].sort_values('date')
    bc = bc.dropna(subset=['beta_excess'])
    ax.plot(bc['date'], bc['beta_excess'],
            linewidth=1.1, color=color, alpha=0.85, label=lbl)

ax.axhline(y=0, color='black', linewidth=0.8)
ax.fill_between(ax.get_xlim(), 0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 0.01,
                alpha=0.04, color='#e74c3c')
ax.fill_between(ax.get_xlim(), ax.get_ylim()[0] if ax.get_ylim()[0] < 0 else -0.01, 0,
                alpha=0.04, color='#3498db')

# Tariff annotation
tariff_date = pd.Timestamp('2025-01-15')
ax.axvline(x=tariff_date, color='#c0392b', linestyle='--', linewidth=0.8, alpha=0.6)
ax.annotate('Tariff risk\nJan 2025', xy=(tariff_date, ax.get_ylim()[1] * 0.85),
            fontsize=8, color='#c0392b', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor='#c0392b', alpha=0.8))

# Annotations for zones
ax.text(0.98, 0.95, 'COMEX Rich / Scarcity \u2191', transform=ax.transAxes,
        fontsize=8, color='#c0392b', ha='right', va='top', alpha=0.6)
ax.text(0.98, 0.05, 'COMEX Cheap / Normalising \u2193', transform=ax.transAxes,
        fontsize=8, color='#2980b9', ha='right', va='bottom', alpha=0.6)

ax.set_title('EFP Beta Excess (Empirical \u2212 Theoretical): Gold vs Silver',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Beta Excess', fontsize=11)
ax.legend(loc='upper left', fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.show()

### Chart 5 — Delta Exposure Heatmaps (Gold & Silver)

In [ ]:
position_sizes = [500, 1_000, 2_500, 5_000, 10_000]
spot_moves_pct = [-3.0, -2.0, -1.0, +1.0, +2.0, +3.0]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for col_idx, metal in enumerate(config['metals']):
    ax = axes[col_idx]

    # Get current empirical beta
    rb_metal = rolling_betas[
        rolling_betas['metal'] == metal
    ].sort_values('date').dropna(subset=['rolling_beta'])

    if len(rb_metal) == 0:
        ax.text(0.5, 0.5, f'No rolling beta for {metal}',
                ha='center', va='center', transform=ax.transAxes)
        continue

    emp_beta = rb_metal['rolling_beta'].iloc[-1]
    latest = master_df[master_df['metal'] == metal].sort_values('date').iloc[-1]
    spot = latest['spot']

    # Build heatmap matrix
    heatmap_data = np.zeros((len(spot_moves_pct), len(position_sizes)))
    for r, pct_move in enumerate(spot_moves_pct):
        for c, pos_oz in enumerate(position_sizes):
            dollar_move = spot * (pct_move / 100.0)
            pnl = pos_oz * emp_beta * dollar_move
            heatmap_data[r, c] = pnl

    heatmap_df = pd.DataFrame(
        heatmap_data,
        index=[f'{m:+.0f}%' for m in spot_moves_pct],
        columns=[f'{p:,}' for p in position_sizes],
    )

    # Diverging colormap centered at zero
    vmax = np.abs(heatmap_data).max()
    sns.heatmap(heatmap_df, annot=True, fmt=',.0f', center=0,
                cmap='RdYlGn', vmin=-vmax, vmax=vmax,
                linewidths=0.5, linecolor='white',
                cbar_kws={'label': 'USD P&L'},
                ax=ax)

    ax.set_title(f'{metal.upper()} — Residual Delta P&L (USD)\n'
                 f'Beta={emp_beta:.4f}, Spot=${spot:,.0f}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Position Size (oz)', fontsize=10)
    ax.set_ylabel('Spot Move (%)', fontsize=10)

plt.tight_layout()
plt.show()

print("Chart 5: Delta exposure heatmaps rendered.")
print("  Green = positive P&L, Red = negative P&L from residual delta")
print("  These are P&L BEYOND carry — the directional residual only.")

## Section 19 — Summary Dashboard

Three formatted tables for quick daily reference:
1. **EFP Beta Summary Statistics** — full-sample regression results
2. **Current EFP Snapshot** — today's live market values
3. **Residual Delta for Standard Position Sizes** — actionable risk numbers

### Table 1 — EFP Beta Summary Statistics

In [ ]:
# ── Table 1: EFP Beta Summary Statistics ─────────────────────────
print("=" * 110)
print("  TABLE 1 — EFP BETA SUMMARY STATISTICS")
print("=" * 110)

header = (f"  {'Metal':8s}  {'Static β(Δ)':>12s}  {'Static β(Lvl)':>14s}  "
          f"{'Theo β(Avg)':>12s}  {'β Excess(Avg)':>14s}  "
          f"{'R²':>6s}  {'HAC t-stat':>10s}  {'Sample':>24s}  {'N':>6s}")
print(header)
print(f"  {'─' * 106}")

for metal in config['metals']:
    sr = static_results.get(metal, {})
    lr = level_results.get(metal, {}) if 'level_results' in dir() else {}

    # Average theoretical beta
    bc_sub = beta_comparison[beta_comparison['metal'] == metal].dropna(
        subset=['beta_theo'])
    avg_theo = bc_sub['beta_theo'].mean() if len(bc_sub) > 0 else np.nan
    avg_excess = bc_sub['beta_excess'].mean() if len(bc_sub) > 0 else np.nan

    beta_d = sr.get('beta_spot', np.nan)
    beta_l = lr.get('beta_level', np.nan)
    r2 = sr.get('r_squared', np.nan)
    t_stat = sr.get('t_stat', np.nan)
    n_obs = sr.get('n_obs', 0)
    dr = sr.get('date_range', (pd.NaT, pd.NaT))

    sample_str = (f"{dr[0]:%Y-%m-%d} to {dr[1]:%Y-%m-%d}"
                  if pd.notna(dr[0]) else "N/A")

    print(f"  {metal.upper():8s}  "
          f"{beta_d:>+12.6f}  "
          f"{beta_l:>+14.6f}  " if pd.notna(beta_l) else f"  {metal.upper():8s}  "
          f"{beta_d:>+12.6f}  "
          f"{'N/A':>14s}  ",
          end="")
    print(f"{avg_theo:>12.6f}  " if pd.notna(avg_theo) else f"{'N/A':>12s}  ", end="")
    print(f"{avg_excess:>+14.6f}  " if pd.notna(avg_excess) else f"{'N/A':>14s}  ", end="")
    print(f"{r2:>6.4f}  " if pd.notna(r2) else f"{'N/A':>6s}  ", end="")
    print(f"{t_stat:>+10.3f}  " if pd.notna(t_stat) else f"{'N/A':>10s}  ", end="")
    print(f"{sample_str:>24s}  {n_obs:>6,}")

### Table 2 — Current EFP Snapshot

In [ ]:
# ── Table 2: Current EFP Snapshot ─────────────────────────────────
print("=" * 140)
print("  TABLE 2 — CURRENT EFP SNAPSHOT")
print("=" * 140)

header2 = (f"  {'Metal':8s}  {'Contract':>10s}  {'DaysFND':>7s}  "
           f"{'Spot':>10s}  {'Futures':>10s}  {'F_OTC':>10s}  "
           f"{'EFP_raw':>10s}  {'EFP_adj':>10s}  {'EFP_ann%':>9s}  "
           f"{'FwdRate%':>8s}  {'RollBeta':>10s}  {'TheoBeta':>10s}")
print(header2)
print(f"  {'─' * 136}")

for metal in config['metals']:
    latest = master_df[master_df['metal'] == metal].sort_values('date').iloc[-1]

    # Latest rolling beta
    rb_m = rolling_betas[
        rolling_betas['metal'] == metal
    ].sort_values('date').dropna(subset=['rolling_beta'])
    rb_val = rb_m['rolling_beta'].iloc[-1] if len(rb_m) > 0 else np.nan

    # Contract name (from ticker)
    contract = latest.get('ticker', 'N/A')

    # Days to FND
    days_fnd = int(latest['delta_T'] * 365) if pd.notna(latest['delta_T']) else 0

    # EFP_adj_ann as %
    ann_pct = (latest['EFP_adj_ann'] * 100
               if pd.notna(latest.get('EFP_adj_ann')) else np.nan)

    # Fwd rate
    fwd_pct = (latest['fwd_rate_interp'] * 100
               if pd.notna(latest.get('fwd_rate_interp')) else np.nan)

    theo = latest.get('beta_theo', np.nan)

    print(f"  {metal.upper():8s}  "
          f"{str(contract):>10s}  "
          f"{days_fnd:>7d}  "
          f"${latest['spot']:>9,.2f}  "
          f"${latest['futures_price']:>9,.2f}  "
          f"${latest['F_OTC']:>9,.2f}  "
          f"${latest['EFP_raw']:>+9.2f}  "
          f"${latest['EFP_adj']:>+9.4f}  ", end="")
    print(f"{ann_pct:>+8.3f}%  " if pd.notna(ann_pct) else f"{'N/A':>9s}  ", end="")
    print(f"{fwd_pct:>+7.3f}%  " if pd.notna(fwd_pct) else f"{'N/A':>8s}  ", end="")
    print(f"{rb_val:>+10.6f}  " if pd.notna(rb_val) else f"{'N/A':>10s}  ", end="")
    print(f"{theo:>10.6f}" if pd.notna(theo) else f"{'N/A':>10s}")

print(f"\n  Snapshot date: {master_df['date'].max():%Y-%m-%d}")

### Table 3 — Residual Delta for Standard Position Sizes

In [ ]:
# ── Table 3: Residual Delta for Standard Positions ───────────────
print("=" * 90)
print("  TABLE 3 — RESIDUAL DELTA FOR STANDARD POSITION SIZES")
print("=" * 90)

print(f"\n  {'Metal':8s}  {'Position (oz)':>14s}  {'Emp Beta':>12s}  "
      f"{'USD Delta / 1% Spot':>22s}")
print(f"  {'─' * 62}")

position_map = {
    'gold':   [1_000, 5_000, 10_000],
    'silver': [5_000, 25_000, 50_000],
}

for metal in config['metals']:
    latest = master_df[master_df['metal'] == metal].sort_values('date').iloc[-1]
    spot = latest['spot']

    rb_m = rolling_betas[
        rolling_betas['metal'] == metal
    ].sort_values('date').dropna(subset=['rolling_beta'])
    emp_beta = rb_m['rolling_beta'].iloc[-1] if len(rb_m) > 0 else np.nan

    for pos in position_map[metal]:
        if pd.notna(emp_beta):
            usd_delta = pos * emp_beta * (spot / 100.0)
            print(f"  {metal.upper():8s}  {pos:>14,}  {emp_beta:>+12.6f}  "
                  f"${usd_delta:>+20,.2f}")
        else:
            print(f"  {metal.upper():8s}  {pos:>14,}  {'N/A':>12s}  {'N/A':>22s}")

print(f"\n  Formula: USD delta = position_oz × beta × (spot / 100)")
print(f"  Interpretation: P&L from a 1% spot move BEYOND carry")

## Section 20 — Sensitivity Analysis

Two sensitivity tests:
1. **Window-length sensitivity**: How stable is rolling beta across
   30, 60, 90, and 120-day estimation windows?
2. **Forward rate sensitivity**: What if the forward rate were
   +50bps or -50bps from observed? Impact on EFP\_adj and beta\_theo.

### Sensitivity 1 — Rolling Window Length

In [ ]:
def efp_beta_sensitivity(metal, master_df, windows=[30, 60, 90, 120]):
    """Run rolling beta for multiple window lengths and plot.

    Parameters
    ----------
    metal : str
        'gold' or 'silver'.
    master_df : pd.DataFrame
        Master DataFrame with delta_spot and delta_efp_adj.
    windows : list of int
        Rolling window sizes in trading days.

    Returns
    -------
    dict
        {window: pd.DataFrame with date and rolling_beta}.
    """
    sub = master_df[
        (master_df['metal'] == metal) &
        (~master_df['exclude_from_regression'].astype(bool))
    ][['date', 'delta_spot', 'delta_efp_adj']].dropna().copy()
    sub = sub.sort_values('date').reset_index(drop=True)

    results = {}
    for w in windows:
        if len(sub) < w + 10:
            continue
        y = sub['delta_efp_adj']
        X = sm.add_constant(sub['delta_spot'])
        rols = RollingOLS(y, X, window=w).fit()
        df = pd.DataFrame({
            'date': sub['date'].values,
            'rolling_beta': rols.params.iloc[:, 1].values,
        })
        results[w] = df

    return results


fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
colors = ['#3498db', '#2c3e50', '#e67e22', '#e74c3c']
windows = [30, 60, 90, 120]

for i, metal in enumerate(config['metals']):
    ax = axes[i]
    sens = efp_beta_sensitivity(metal, master_df, windows=windows)

    for j, (w, df) in enumerate(sens.items()):
        df_clean = df.dropna(subset=['rolling_beta'])
        lw = 1.5 if w == 60 else 0.9
        alpha = 1.0 if w == 60 else 0.6
        ax.plot(df_clean['date'], df_clean['rolling_beta'],
                linewidth=lw, color=colors[j], alpha=alpha,
                label=f'{w}d window')

    ax.axhline(y=0, color='grey', linestyle='--', linewidth=0.5)
    ax.set_title(f'{metal.upper()} — Rolling Beta Sensitivity to Window Length',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Rolling Beta', fontsize=10)
    ax.legend(loc='best', fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.show()

print("Window sensitivity analysis complete.")
print("  60d window is the default; shorter = noisier, longer = smoother/laggier")

### Sensitivity 2 — Forward Rate Shock (+/- 50bps)

In [ ]:
# ── Forward rate sensitivity ──────────────────────────────────────
shocks_bps = [-50, 0, +50]

print("=" * 90)
print("  FORWARD RATE SENSITIVITY: Impact of +/- 50bps on EFP_adj and beta_theo")
print("=" * 90)

print(f"\n  {'Metal':8s}  {'Shock':>8s}  {'Fwd Rate%':>10s}  "
      f"{'F_OTC':>12s}  {'EFP_adj':>12s}  {'beta_theo':>12s}")
print(f"  {'─' * 68}")

for metal in config['metals']:
    latest = master_df[master_df['metal'] == metal].sort_values('date').iloc[-1]
    spot = latest['spot']
    delta_T = latest['delta_T']
    fwd_rate = latest['fwd_rate_interp']
    futures = latest['futures_price']

    for shock in shocks_bps:
        shocked_rate = fwd_rate + shock / 10_000  # bps to decimal
        shocked_fotc = spot * (1.0 + shocked_rate * delta_T)
        shocked_efp_adj = futures - shocked_fotc
        shocked_beta_theo = shocked_rate * delta_T

        label = f"{shock:+d}bps" if shock != 0 else "BASE"
        print(f"  {metal.upper():8s}  {label:>8s}  "
              f"{shocked_rate*100:>+9.4f}%  "
              f"${shocked_fotc:>11,.2f}  "
              f"${shocked_efp_adj:>+11.4f}  "
              f"{shocked_beta_theo:>12.6f}")

print(f"\n  Note: A +50bps rate shock reduces F_OTC (making EFP_adj more positive)")
print(f"  and increases beta_theo. This shows how sensitive the EFP is to")
print(f"  the accuracy of the forward rate curve.")

## Section 21 — HOW TO USE THIS NOTEBOOK: Daily Workflow

### What this notebook does

This notebook pulls live COMEX gold and silver futures prices, spot prices,
and Bloomberg native forward rate curves via BQL, constructs the EFP
(Exchange for Physical = COMEX Futures - OTC Forward), and quantifies the
residual spot delta embedded in an EFP position. It auto-updates every time
you run it, always reflecting today's market.

---

### Daily Steps

| Step | Action | Section |
|------|--------|---------|
| 1 | **Re-run Sections 2-5** BQL pulls to refresh all data | Sections 2-5 |
| 2 | **Check Section 6** — has the front month rolled overnight? Look for sawtooth jumps in delta\_T | Section 6 |
| 3 | **Review Section 12 charts** — is EFP\_adj normal or spiking? Spikes = physical scarcity / tariff premium | Section 12 |
| 4 | **Read Table 2 (Section 19)** — today's live EFP snapshot: spot, futures, F\_OTC, EFP\_adj, rolling beta | Section 19 |
| 5 | **Read current rolling beta** — what directional delta are you carrying beyond carry? | Section 14, Table 2 |
| 6 | **Use Table 3** to determine any delta hedge required for your position size | Section 19, Table 3 |

---

### Parameter Tuning Guide

| Parameter | Location | Default | Effect |
|-----------|----------|---------|--------|
| `regression_window_days` | `config` dict (Section 0) | 60 | Rolling OLS window. Shorter = more reactive, noisier. Longer = smoother, laggier. |
| `roll_days_before_fnd` | `config` dict (Section 0) | 5 | Days before FND to skip in roll detection. Increase if you see spurious roll flags near expiry. |
| `start_date` | `config` dict (Section 0) | 2023-01-01 | Set earlier for longer history (e.g. 2020-01-01). May slow BQL pulls. |
| `k_sigma` | `config` dict (Section 0) | 3 | Roll detection threshold (multiples of 20d rolling std). Lower = more sensitive. |

---

### Interpretation Guide

| Signal | Meaning | Action |
|--------|---------|--------|
| **EFP\_adj > 0** | COMEX rich to OTC; physical scarcity or tariff/repatriation premium priced in | Monitor for widening; consider selling EFP if premium is extreme |
| **EFP\_adj < 0** | COMEX cheap to OTC; rare, possible cross-market liquidity dislocation | Potential buying opportunity |
| **beta\_excess > 0** | Residual delta of long EFP larger than pure carry theory predicts; net long bias stronger than expected | Hedge additional delta if unwanted; or let ride if directionally bullish |
| **beta\_excess < 0** | EFP moves less than theory suggests; possible mean-reversion or offsetting rate move | Less hedging needed; EFP is a purer carry trade |
| **Rolling beta unstable** | Regime change (compare 30d vs 120d in Section 20) | Use wider window or reduce position size |

## Section 23 — End-to-End Validation: Worked Example

> **PURPOSE:** Verify the entire pipeline computes correctly by walking
> through every step on a single known historical date.
>
> We pick a date in **January 2025** when gold EFPs were elevated due to
> tariff concerns — this makes the validation meaningful because the EFP
> was non-trivially different from zero.

In [ ]:
# ── Pick a validation date ─────────────────────────────────────────
# Target: a date in Jan 2025 when gold EFP was elevated
gold_sub = master_df[master_df['metal'] == 'gold'].sort_values('date')

# Find a date near 2025-01-15 (tariff risk period)
target = pd.Timestamp('2025-01-15')
gold_sub['_dist'] = (gold_sub['date'] - target).abs()
val_idx = gold_sub['_dist'].idxmin()
val_row = gold_sub.loc[val_idx].copy()
val_date = val_row['date']

print("=" * 80)
print("  END-TO-END VALIDATION — WORKED EXAMPLE")
print("  Verifies the entire pipeline on a known historical date")
print("=" * 80)

print(f"\n  Validation date: {val_date:%Y-%m-%d}")
print(f"  (Selected as nearest available date to 2025-01-15, tariff period)")

# Step 1: Active contract
print(f"\n  1. Active front-month (GC1): {val_row['ticker']}")

# Step 2: FND and delta_T
print(f"  2. FND: {val_row['fnd']}")
print(f"     delta_T: {val_row['delta_T']:.6f} years "
      f"({val_row['delta_T']*365:.1f} calendar days)")

# Step 3: Spot price
print(f"  3. Spot price (XAU): ${val_row['spot']:,.2f}")

# Step 4: Futures price
print(f"  4. Futures price (GC1): ${val_row['futures_price']:,.2f}")

# Step 5: Forward rate tenors
print(f"  5. Bloomberg forward rate tenors:")
fwd_cols = ['fwd_1W', 'fwd_1M', 'fwd_2M', 'fwd_3M', 'fwd_6M', 'fwd_12M']
tenor_labels = ['1W', '1M', '2M', '3M', '6M', '12M']
for col, lbl in zip(fwd_cols, tenor_labels):
    val = val_row.get(col, np.nan)
    if pd.notna(val):
        print(f"     {lbl:>4s}: {val*100:+.4f}%")
    else:
        print(f"     {lbl:>4s}: N/A")

# Step 6: Interpolated forward rate
print(f"  6. Interpolated forward rate at delta_T={val_row['delta_T']:.4f}y: "
      f"{val_row['fwd_rate_interp']*100:+.4f}%")

# Step 7: F_OTC
# Recompute to verify
f_otc_check = val_row['spot'] * (1.0 + val_row['fwd_rate_interp'] * val_row['delta_T'])
print(f"  7. F_OTC = spot × (1 + fwd_rate × delta_T)")
print(f"         = {val_row['spot']:,.2f} × (1 + {val_row['fwd_rate_interp']:.6f} "
      f"× {val_row['delta_T']:.6f})")
print(f"         = ${f_otc_check:,.4f}")
print(f"     Stored: ${val_row['F_OTC']:,.4f}  "
      f"(diff: ${abs(f_otc_check - val_row['F_OTC']):.6f})")

# Step 8: EFP_raw
efp_raw_check = val_row['futures_price'] - val_row['spot']
print(f"  8. EFP_raw = futures - spot = "
      f"${val_row['futures_price']:,.2f} - ${val_row['spot']:,.2f} "
      f"= ${efp_raw_check:+,.2f}")
print(f"     Stored: ${val_row['EFP_raw']:+,.2f}")

# Step 9: EFP_adj
efp_adj_check = val_row['futures_price'] - f_otc_check
print(f"  9. EFP_adj = futures - F_OTC = "
      f"${val_row['futures_price']:,.2f} - ${f_otc_check:,.4f} "
      f"= ${efp_adj_check:+,.4f}")
print(f"     Stored: ${val_row['EFP_adj']:+,.4f}")

# Step 10: Theoretical beta
beta_theo_check = val_row['fwd_rate_interp'] * val_row['delta_T']
print(f"  10. beta_theo = fwd_rate_interp × delta_T = "
      f"{val_row['fwd_rate_interp']:.6f} × {val_row['delta_T']:.6f} "
      f"= {beta_theo_check:.6f}")
print(f"      Stored: {val_row.get('beta_theo', np.nan):.6f}")

# Step 11: Rolling empirical beta on that date
rb_gold = rolling_betas[rolling_betas['metal'] == 'gold'].copy()
rb_gold['_dist'] = (rb_gold['date'] - val_date).abs()
if len(rb_gold) > 0:
    rb_nearest = rb_gold.loc[rb_gold['_dist'].idxmin()]
    print(f"  11. Rolling empirical beta (60d) on {rb_nearest['date']:%Y-%m-%d}: "
          f"{rb_nearest['rolling_beta']:+.6f}")
    emp_beta_val = rb_nearest['rolling_beta']
else:
    print(f"  11. Rolling empirical beta: N/A (insufficient data)")
    emp_beta_val = np.nan

# Step 12: Beta excess
if pd.notna(emp_beta_val):
    beta_excess_check = emp_beta_val - beta_theo_check
    print(f"  12. beta_excess = empirical - theoretical = "
          f"{emp_beta_val:.6f} - {beta_theo_check:.6f} "
          f"= {beta_excess_check:+.6f}")
else:
    print(f"  12. beta_excess: N/A")
    beta_excess_check = np.nan

# Step 13: USD delta for 10,000 oz
position = 10_000
if pd.notna(emp_beta_val):
    usd_delta = position * emp_beta_val * (val_row['spot'] / 100.0)
    print(f"  13. USD delta exposure for {position:,} oz long EFP, 1% spot move:")
    print(f"      = {position:,} × {emp_beta_val:.6f} × "
          f"(${val_row['spot']:,.2f} / 100)")
    print(f"      = ${usd_delta:+,.2f}")
else:
    print(f"  13. USD delta: N/A")

# Verification summary
print(f"\n  {'─' * 60}")
print(f"  VALIDATION RESULT:")
f_otc_ok = abs(f_otc_check - val_row['F_OTC']) < 0.01
efp_raw_ok = abs(efp_raw_check - val_row['EFP_raw']) < 0.01
efp_adj_ok = abs(efp_adj_check - val_row['EFP_adj']) < 0.01
bt_ok = abs(beta_theo_check - val_row.get('beta_theo', beta_theo_check)) < 0.000001

checks = {
    'F_OTC recomputed matches stored': f_otc_ok,
    'EFP_raw recomputed matches stored': efp_raw_ok,
    'EFP_adj recomputed matches stored': efp_adj_ok,
    'beta_theo recomputed matches stored': bt_ok,
}
for name, ok in checks.items():
    print(f"    {'PASS' if ok else 'FAIL'} | {name}")
print(f"  All {len(checks)} checks passed: {all(checks.values())}")

# Clean up temp columns
gold_sub.drop(columns=['_dist'], inplace=True, errors='ignore')

---
## Notebook Complete

All 23 sections (Prompts A–D) are implemented. This notebook is
**production-ready** for daily use on Bloomberg BQuant.

**Outputs saved:**
- `efp_master_data.csv` — raw master DataFrame (Prompt A)
- `efp_with_spreads.csv` — with EFP columns (Prompt B)
- `efp_beta_results.csv` — beta comparison series (Prompt C)

In [ ]:
print("=" * 80)
print("  NOTEBOOK COMPLETE")
print("=" * 80)
print(f"  Timestamp      : {datetime.now():%Y-%m-%d %H:%M:%S}")
print(f"  master_df      : {master_df.shape[0]:,} rows × {master_df.shape[1]} cols")
print(f"  beta_comparison: {beta_comparison.shape[0]:,} rows × {beta_comparison.shape[1]} cols")
print(f"  rolling_betas  : {rolling_betas.shape[0]:,} rows × {rolling_betas.shape[1]} cols")
print(f"\n  Sections: 0-23 (Prompts A through D)")
print(f"  All data sourced from Bloomberg BQL — auto-updating on re-run.")
print(f"\n  Files saved:")
print(f"    efp_master_data.csv")
print(f"    efp_with_spreads.csv")
print(f"    efp_beta_results.csv")